# Multi-Task SegFormer — Training Pipeline

Full training and Optuna hyperparameter optimization (HPO) pipeline for the proposed Multi-Task SegFormer model.

**Manuscript reference:** Table 1 (SegFormer rows), Section 2.2.2

The cell outputs below show the actual training logs produced during the experiments reported in the paper.

In [1]:
import os
import glob
import random
import csv
import json
import time
from collections import Counter

import numpy as np
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import (
    AutoImageProcessor,
    SegformerForSemanticSegmentation,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve, auc, f1_score
import optuna


torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

NUM_SEG_LABELS = 2
NUM_CLS_LABELS = 2


def set_seed(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))


def list_image_files(dir_path):
    exts = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
    out = []
    for p in sorted(glob.glob(os.path.join(dir_path, "*"))):
        if os.path.isfile(p) and p.lower().endswith(exts):
            out.append(p)
    return out


def get_patient_id_from_path(image_path):
    base = os.path.basename(image_path)
    name, _ = os.path.splitext(base)
    if name.startswith("image_"):
        return name[len("image_") :]
    return name


def get_patient_group_from_id(pid):
    if "C" in pid:
        return "control"
    if "P" in pid:
        return "arthrit"
    return "unknown"


def get_patient_base_id(pid):
    if "_" in pid:
        return pid.rsplit("_", 1)[0]
    return pid


def get_cls_label_from_path(image_path):
    pid = get_patient_id_from_path(image_path)
    g = get_patient_group_from_id(pid)
    if g == "control":
        return 0
    if g == "arthrit":
        return 1
    return -100


def resolve_mask_path(image_path):
    img_dir = os.path.dirname(image_path)
    parent = os.path.dirname(img_dir)
    mask_dir = os.path.join(parent, "ROI")
    base = os.path.basename(image_path)
    name, ext = os.path.splitext(base)
    if name.startswith("image_"):
        suffix = name[len("image_") :]
        mask_name = "mask_" + suffix + ext
    else:
        if base.startswith("image"):
            mask_name = "mask" + base[5:]
        else:
            mask_name = "mask_" + base
    mask_path = os.path.join(mask_dir, mask_name)
    if not os.path.exists(mask_path):
        raise FileNotFoundError(f"Mask not found for {image_path}: {mask_path}")
    return mask_path


def load_mask_array(image_path):
    mask_path = resolve_mask_path(image_path)
    mask = Image.open(mask_path)
    mask = np.array(mask)
    if mask.ndim == 3:
        mask = mask[:, :, 0]
    if mask.max() > 1:
        mask = (mask > 0).astype("int64")
    else:
        mask = mask.astype("int64")
    return mask


class USGSegmentationClassificationDataset(Dataset):
    def __init__(self, image_paths, processor, include_cls_labels=False):
        self.image_paths = list(image_paths)
        self.processor = processor
        self.include_cls_labels = bool(include_cls_labels)

        base_ids = [get_patient_base_id(get_patient_id_from_path(p)) for p in self.image_paths]
        uniq = sorted(list(set(base_ids)))
        self.base2idx = {b: i for i, b in enumerate(uniq)}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[int(idx)]
        image = Image.open(image_path).convert("RGB")
        mask = load_mask_array(image_path)

        encoded = self.processor(images=image, segmentation_maps=[mask], return_tensors="pt")
        pixel_values = encoded["pixel_values"].squeeze(0)
        labels = encoded["labels"].squeeze(0)

        pid = get_patient_id_from_path(image_path)
        base_id = get_patient_base_id(pid)
        patient_idx = torch.tensor(self.base2idx[base_id], dtype=torch.long)

        item = {"pixel_values": pixel_values, "labels": labels, "patient_idx": patient_idx}

        if self.include_cls_labels:
            cls_label = get_cls_label_from_path(image_path)
            item["cls_labels"] = torch.tensor(cls_label, dtype=torch.long)

        return item


def stratified_split_base_ids(base_ids, val_ratio, rng):
    control_ids = []
    arthrit_ids = []
    other_ids = []

    for bid in base_ids:
        g = get_patient_group_from_id(bid)
        if g == "control":
            control_ids.append(bid)
        elif g == "arthrit":
            arthrit_ids.append(bid)
        else:
            other_ids.append(bid)

    rng.shuffle(control_ids)
    rng.shuffle(arthrit_ids)
    rng.shuffle(other_ids)

    def split_group(ids):
        if len(ids) <= 1:
            return ids, []
        split_index = int(len(ids) * (1.0 - float(val_ratio)))
        if split_index <= 0:
            split_index = 1
        if split_index >= len(ids):
            split_index = len(ids) - 1
        train_g = ids[:split_index]
        val_g = ids[split_index:]
        return train_g, val_g

    train_c, val_c = split_group(control_ids)
    train_a, val_a = split_group(arthrit_ids)
    train_o, val_o = split_group(other_ids)

    train_ids = train_c + train_a + train_o
    val_ids = val_c + val_a + val_o

    stats = {
        "train_control": len(train_c),
        "train_arthrit": len(train_a),
        "val_control": len(val_c),
        "val_arthrit": len(val_a),
    }
    return train_ids, val_ids, stats


def split_patients(train_val_usg_dir, val_ratio, seed):
    if not os.path.isdir(train_val_usg_dir):
        raise RuntimeError(f"Directory not found: {train_val_usg_dir}")

    all_image_paths = list_image_files(train_val_usg_dir)
    if len(all_image_paths) == 0:
        raise RuntimeError(f"No images found in {train_val_usg_dir}")

    base_to_images = {}
    for p in all_image_paths:
        pid = get_patient_id_from_path(p)
        bid = get_patient_base_id(pid)
        base_to_images.setdefault(bid, []).append(p)

    base_ids = list(base_to_images.keys())
    rng = np.random.RandomState(int(seed))
    train_bids, val_bids, stats = stratified_split_base_ids(base_ids, val_ratio, rng)

    if not set(train_bids).isdisjoint(set(val_bids)):
        raise RuntimeError("Base patient leakage between train and val sets")

    train_paths = []
    val_paths = []
    for bid in train_bids:
        train_paths.extend(base_to_images[bid])
    for bid in val_bids:
        val_paths.extend(base_to_images[bid])

    print(
        f"Train base patients: {len(train_bids)}, images: {len(train_paths)}, "
        f"control: {stats['train_control']}, arthrit: {stats['train_arthrit']}"
    )
    print(
        f"Val base patients: {len(val_bids)}, images: {len(val_paths)}, "
        f"control: {stats['val_control']}, arthrit: {stats['val_arthrit']}"
    )

    return train_paths, val_paths


def get_fold_test_image_paths(fold_root):
    test_usg_dir = os.path.join(fold_root, "test", "darkened")
    test_roi_dir = os.path.join(fold_root, "test", "ROI")
    if not os.path.isdir(test_usg_dir):
        raise RuntimeError(f"Test images dir not found: {test_usg_dir}")
    if not os.path.isdir(test_roi_dir):
        raise RuntimeError(f"Test ROI dir not found: {test_roi_dir}")
    all_image_paths = list_image_files(test_usg_dir)
    if len(all_image_paths) == 0:
        raise RuntimeError(f"No test images found in {test_usg_dir}")
    return all_image_paths


def get_external_test_image_paths(external_test_usg_dir):
    if external_test_usg_dir is None:
        return []
    if not os.path.isdir(external_test_usg_dir):
        return []
    return list_image_files(external_test_usg_dir)


def make_patient_balanced_sampler(image_paths, seed):
    pids = []
    for p in image_paths:
        pid = get_patient_id_from_path(p)
        pids.append(get_patient_base_id(pid))
    cnt = Counter(pids)
    weights = torch.tensor([1.0 / float(cnt[pid]) for pid in pids], dtype=torch.double)
    g = torch.Generator()
    g.manual_seed(int(seed))
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True, generator=g)


def _safe_auc(y_true, y_score):
    if len(y_true) == 0:
        return 0.0, 0.0
    if len(np.unique(y_true)) <= 1:
        return 0.0, 0.0
    try:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc_v = float(auc(fpr, tpr))
    except Exception:
        roc_auc_v = 0.0
    try:
        prec, rec, _ = precision_recall_curve(y_true, y_score)
        pr_auc_v = float(auc(rec, prec))
    except Exception:
        pr_auc_v = 0.0
    return roc_auc_v, pr_auc_v


def compute_metrics(eval_pred):
    if hasattr(eval_pred, "predictions"):
        preds = eval_pred.predictions
        labels = eval_pred.label_ids
    else:
        preds, labels = eval_pred

    seg_logits_np = None
    cls_logits_np = None
    if isinstance(preds, (tuple, list)) and len(preds) >= 1:
        seg_logits_np = preds[0]
        if len(preds) >= 2:
            cls_logits_np = preds[1]
    else:
        seg_logits_np = preds

    seg_labels_np = None
    cls_labels_np = None
    patient_idx_np = None
    if isinstance(labels, (tuple, list)) and len(labels) >= 1:
        seg_labels_np = labels[0]
        if len(labels) >= 2:
            cls_labels_np = labels[1]
        if len(labels) >= 3:
            patient_idx_np = labels[2]
    else:
        seg_labels_np = labels

    out = {}

    if seg_logits_np is not None and seg_labels_np is not None:
        logits = torch.as_tensor(seg_logits_np)
        logits = torch.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0).clamp(-50.0, 50.0)
        labels_t = torch.as_tensor(seg_labels_np).long()

        upsampled_logits = F.interpolate(
            logits,
            size=labels_t.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        pred_mask = upsampled_logits.argmax(dim=1)

        valid = labels_t != 255
        pred_v = pred_mask[valid]
        lab_v = labels_t[valid]

        ious = []
        dices = []
        for label_id in range(NUM_SEG_LABELS):
            p = pred_v == label_id
            t = lab_v == label_id
            inter = (p & t).sum().item()
            union = (p | t).sum().item()
            denom = p.sum().item() + t.sum().item()
            iou = float(inter) / float(union) if union != 0 else 0.0
            dice = (2.0 * float(inter) / float(denom)) if denom != 0 else 0.0
            ious.append(iou)
            dices.append(dice)

        out["mIoU_image"] = float(np.mean(ious))
        out["mDice_image"] = float(np.mean(dices))
        out["pixel_accuracy_image"] = float((pred_v == lab_v).float().mean().item())
        out["mIoU_fg_image"] = float(np.mean(ious[1:])) if NUM_SEG_LABELS > 1 else float(ious[0])
        out["mDice_fg_image"] = float(np.mean(dices[1:])) if NUM_SEG_LABELS > 1 else float(dices[0])
        out["iou_class_0_image"] = float(ious[0])
        out["iou_class_1_image"] = float(ious[1])
        out["dice_class_0_image"] = float(dices[0])
        out["dice_class_1_image"] = float(dices[1])

        y_true_flat = lab_v.view(-1).cpu().numpy()
        y_pred_flat = pred_v.view(-1).cpu().numpy()
        try:
            out["f1_micro_image"] = float(f1_score(y_true_flat, y_pred_flat, average="micro"))
            out["f1_macro_image"] = float(f1_score(y_true_flat, y_pred_flat, average="macro"))
            out["f1_fluid_image"] = float(
                f1_score((y_true_flat == 1).astype(np.int32), (y_pred_flat == 1).astype(np.int32))
            )
        except Exception:
            out["f1_micro_image"] = 0.0
            out["f1_macro_image"] = 0.0
            out["f1_fluid_image"] = 0.0

        if patient_idx_np is not None:
            pidx = torch.as_tensor(patient_idx_np).long().cpu().numpy()
            pred_np = pred_mask.detach().cpu().numpy()
            lab_np = labels_t.detach().cpu().numpy()

            uniq_p = np.unique(pidx)
            per_p_miou = []
            per_p_mdice = []
            per_p_miou_fg = []
            per_p_mdice_fg = []
            per_p_acc = []
            per_p_iou_c0 = []
            per_p_iou_c1 = []
            per_p_dice_c0 = []
            per_p_dice_c1 = []

            for pid in uniq_p:
                idxs = np.where(pidx == pid)[0]
                inter = np.zeros(NUM_SEG_LABELS, dtype=np.float64)
                union = np.zeros(NUM_SEG_LABELS, dtype=np.float64)
                denom = np.zeros(NUM_SEG_LABELS, dtype=np.float64)
                correct = 0.0
                total = 0.0

                for i in idxs:
                    vmask = lab_np[i] != 255
                    gt = lab_np[i][vmask].astype(np.int64)
                    pr = pred_np[i][vmask].astype(np.int64)
                    if gt.size == 0:
                        continue
                    correct += float((gt == pr).sum())
                    total += float(gt.size)
                    for c in range(NUM_SEG_LABELS):
                        pc = pr == c
                        tc = gt == c
                        inter[c] += float(np.logical_and(pc, tc).sum())
                        union[c] += float(np.logical_or(pc, tc).sum())
                        denom[c] += float(pc.sum() + tc.sum())

                iou_c = np.where(union > 0, inter / union, 0.0)
                dice_c = np.where(denom > 0, (2.0 * inter) / denom, 0.0)

                per_p_miou.append(float(iou_c.mean()))
                per_p_mdice.append(float(dice_c.mean()))
                if NUM_SEG_LABELS > 1:
                    per_p_miou_fg.append(float(iou_c[1:].mean()))
                    per_p_mdice_fg.append(float(dice_c[1:].mean()))
                else:
                    per_p_miou_fg.append(float(iou_c[0]))
                    per_p_mdice_fg.append(float(dice_c[0]))

                acc_p = float(correct / total) if total > 0 else 0.0
                per_p_acc.append(acc_p)

                per_p_iou_c0.append(float(iou_c[0]))
                per_p_iou_c1.append(float(iou_c[1]) if NUM_SEG_LABELS > 1 else 0.0)
                per_p_dice_c0.append(float(dice_c[0]))
                per_p_dice_c1.append(float(dice_c[1]) if NUM_SEG_LABELS > 1 else 0.0)

            out["mIoU_patient"] = float(np.mean(per_p_miou)) if len(per_p_miou) else 0.0
            out["mDice_patient"] = float(np.mean(per_p_mdice)) if len(per_p_mdice) else 0.0
            out["pixel_accuracy_patient"] = float(np.mean(per_p_acc)) if len(per_p_acc) else 0.0
            out["mIoU_fg_patient"] = float(np.mean(per_p_miou_fg)) if len(per_p_miou_fg) else 0.0
            out["mDice_fg_patient"] = float(np.mean(per_p_mdice_fg)) if len(per_p_mdice_fg) else 0.0
            out["iou_class_0_patient"] = float(np.mean(per_p_iou_c0)) if len(per_p_iou_c0) else 0.0
            out["iou_class_1_patient"] = float(np.mean(per_p_iou_c1)) if len(per_p_iou_c1) else 0.0
            out["dice_class_0_patient"] = float(np.mean(per_p_dice_c0)) if len(per_p_dice_c0) else 0.0
            out["dice_class_1_patient"] = float(np.mean(per_p_dice_c1)) if len(per_p_dice_c1) else 0.0

            try:
                out["f1_micro_patient"] = float(
                    np.mean([
                        f1_score(
                            (lab_np[i][lab_np[i] != 255].reshape(-1) == 1).astype(np.int32),
                            (pred_np[i][lab_np[i] != 255].reshape(-1) == 1).astype(np.int32),
                        )
                        for i in range(pred_np.shape[0])
                        if (lab_np[i] != 255).sum() > 0
                    ])
                )
            except Exception:
                out["f1_micro_patient"] = 0.0

    if cls_logits_np is not None and cls_labels_np is not None:
        cls_logits_t = torch.as_tensor(cls_logits_np)
        cls_logits_t = torch.nan_to_num(cls_logits_t, nan=0.0, posinf=50.0, neginf=-50.0).clamp(-50.0, 50.0)
        cls_labels_t = torch.as_tensor(cls_labels_np).long()

        valid_c = cls_labels_t != -100
        if valid_c.any().item():
            logits_v = cls_logits_t[valid_c]
            labels_v = cls_labels_t[valid_c]

            probs = torch.softmax(logits_v, dim=1)
            probs = torch.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
            row_sums = probs.sum(dim=1, keepdim=True).clamp_min(1e-12)
            probs = probs / row_sums

            pred = torch.argmax(probs, dim=1)

            y_true = labels_v.cpu().numpy()
            y_pred = pred.cpu().numpy()
            y_score = probs[:, 1].detach().cpu().numpy()

            finite = np.isfinite(y_score)
            y_true_f = y_true[finite]
            y_pred_f = y_pred[finite]
            y_score_f = y_score[finite]

            if len(y_true_f) == 0:
                out["cls_acc_image"] = 0.0
                out["cls_f1_image"] = 0.0
                out["cls_roc_auc_image"] = 0.0
                out["cls_pr_auc_image"] = 0.0
            else:
                out["cls_acc_image"] = float((y_pred_f == y_true_f).mean())
                try:
                    out["cls_f1_image"] = float(f1_score(y_true_f, y_pred_f))
                except Exception:
                    out["cls_f1_image"] = 0.0
                roc_auc_v, pr_auc_v = _safe_auc(y_true_f, y_score_f)
                out["cls_roc_auc_image"] = float(roc_auc_v)
                out["cls_pr_auc_image"] = float(pr_auc_v)

            if patient_idx_np is not None:
                pidx_full = torch.as_tensor(patient_idx_np).long().cpu().numpy()
                pidx_v = pidx_full[valid_c.cpu().numpy()]

                y_true_v = y_true_f
                y_score_v = y_score_f

                uniq_p = np.unique(pidx_v)
                p_true = []
                p_score = []

                for pid in uniq_p:
                    ids = np.where(pidx_v == pid)[0]
                    if len(ids) == 0:
                        continue
                    y_t = y_true_v[ids]
                    y_s = y_score_v[ids]
                    if len(y_t) == 0:
                        continue
                    if np.all(y_t == y_t[0]):
                        gt = int(y_t[0])
                    else:
                        gt = int(np.round(np.mean(y_t)))
                    sc = float(np.mean(y_s))
                    if not np.isfinite(sc):
                        continue
                    p_true.append(gt)
                    p_score.append(sc)

                if len(p_true) == 0:
                    out["cls_acc_patient"] = 0.0
                    out["cls_f1_patient"] = 0.0
                    out["cls_roc_auc_patient"] = 0.0
                    out["cls_pr_auc_patient"] = 0.0
                else:
                    p_true_np = np.array(p_true, dtype=np.int64)
                    p_score_np = np.array(p_score, dtype=np.float64)
                    p_pred_np = (p_score_np >= 0.5).astype(np.int64)

                    out["cls_acc_patient"] = float((p_pred_np == p_true_np).mean())
                    try:
                        out["cls_f1_patient"] = float(f1_score(p_true_np, p_pred_np))
                    except Exception:
                        out["cls_f1_patient"] = 0.0
                    roc_auc_v, pr_auc_v = _safe_auc(p_true_np, p_score_np)
                    out["cls_roc_auc_patient"] = float(roc_auc_v)
                    out["cls_pr_auc_patient"] = float(pr_auc_v)
        else:
            out["cls_acc_image"] = 0.0
            out["cls_f1_image"] = 0.0
            out["cls_roc_auc_image"] = 0.0
            out["cls_pr_auc_image"] = 0.0
            out["cls_acc_patient"] = 0.0
            out["cls_f1_patient"] = 0.0
            out["cls_roc_auc_patient"] = 0.0
            out["cls_pr_auc_patient"] = 0.0

    return out


def compute_seg_class_weights(image_paths):
    counts = np.zeros(NUM_SEG_LABELS, dtype=np.float64)
    for p in image_paths:
        m = load_mask_array(p)
        for c in range(NUM_SEG_LABELS):
            counts[c] += (m == c).sum()
    total = float(counts.sum())
    if total <= 0:
        return np.ones(NUM_SEG_LABELS, dtype=np.float32)
    freq = counts / total
    weights = 1.0 / (freq + 1e-6)
    weights = weights / weights.sum() * NUM_SEG_LABELS
    return weights.astype(np.float32)


def compute_cls_class_weights(image_paths, control_boost=1.0, arthrit_boost=1.0):
    counts = np.zeros(NUM_CLS_LABELS, dtype=np.float64)
    for p in image_paths:
        lab = get_cls_label_from_path(p)
        if 0 <= lab < NUM_CLS_LABELS:
            counts[lab] += 1.0
    total = float(counts.sum())
    if total == 0:
        return None
    freq = counts / total
    base_weights = 1.0 / (freq + 1e-6)
    boost = np.array([float(control_boost), float(arthrit_boost)], dtype=np.float64)
    weights = base_weights * boost
    weights = weights / weights.sum() * NUM_CLS_LABELS
    return weights.astype(np.float32)


def segmentation_loss(logits, labels, class_weights=None, focal_gamma=None):
    logits = F.interpolate(
        logits,
        size=labels.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )
    labels = labels.long()
    valid_mask = labels != 255
    logits_v = logits.permute(0, 2, 3, 1)[valid_mask]
    labels_v = labels[valid_mask]

    weight = class_weights.to(logits_v.device) if class_weights is not None else None
    ce_loss = F.cross_entropy(logits_v, labels_v, weight=weight, reduction="none")
    ce_loss_mean = ce_loss.mean()

    focal_term = 0.0
    if focal_gamma is not None:
        num_classes = logits_v.shape[-1]
        log_probs = F.log_softmax(logits_v, dim=-1)
        probs = torch.exp(log_probs)
        targets_one_hot = F.one_hot(labels_v, num_classes=num_classes).float()
        pt = (probs * targets_one_hot).sum(dim=-1)
        focal_term = ((1.0 - pt) ** float(focal_gamma) * ce_loss).mean()

    num_classes = logits_v.shape[-1]
    probs = torch.softmax(logits_v, dim=-1)
    targets_one_hot = F.one_hot(labels_v, num_classes=num_classes).float()
    probs_flat = probs.view(-1, num_classes)
    targets_flat = targets_one_hot.view(-1, num_classes)
    intersection = (probs_flat * targets_flat).sum(dim=0)
    union = probs_flat.sum(dim=0) + targets_flat.sum(dim=0)
    dice_score = (2.0 * intersection + 1.0) / (union + 1.0)
    dice_loss = 1.0 - dice_score.mean()

    return ce_loss_mean + dice_loss + focal_term


def visualize_examples(model, processor, image_paths, output_dir, max_examples=None):
    if len(image_paths) == 0:
        return
    figures_dir = os.path.join(output_dir, "figures")
    os.makedirs(figures_dir, exist_ok=True)
    device = next(model.parameters()).device
    model.eval()
    subset = image_paths if max_examples is None else image_paths[: int(max_examples)]
    for idx, image_path in enumerate(subset):
        image = Image.open(image_path).convert("RGB")
        mask = load_mask_array(image_path)
        inputs = processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)
        with torch.no_grad():
            outputs = model(pixel_values=pixel_values, labels=None, cls_labels=None)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
        upsampled_logits = F.interpolate(
            logits,
            size=mask.shape,
            mode="bilinear",
            align_corners=False,
        )
        pred = upsampled_logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)
        img_np = np.array(image)
        h, w, _ = img_np.shape
        gt_color = np.zeros_like(img_np)
        gt_color[mask == 1] = np.array([255, 0, 0], dtype=np.uint8)
        pred_color = np.zeros_like(img_np)
        pred_color[pred == 1] = np.array([0, 255, 0], dtype=np.uint8)
        overlay_gt = (0.6 * img_np + 0.4 * gt_color).astype(np.uint8)
        overlay_pred = (0.6 * img_np + 0.4 * pred_color).astype(np.uint8)
        canvas = np.zeros((h, w * 3, 3), dtype=np.uint8)
        canvas[:, :w] = img_np
        canvas[:, w : 2 * w] = overlay_gt
        canvas[:, 2 * w : 3 * w] = overlay_pred
        out_img = Image.fromarray(canvas)
        out_path = os.path.join(figures_dir, f"example_{idx + 1}.png")
        out_img.save(out_path)


class EpochMetricsPrinter(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return control
        if "epoch" not in logs:
            return control
        epoch = logs["epoch"]
        keys = [
            "loss",
            "eval_loss",
            "eval_mDice_fg_patient",
            "eval_mIoU_fg_patient",
            "eval_pixel_accuracy_patient",
            "eval_mDice_fg_image",
            "eval_mIoU_fg_image",
            "eval_pixel_accuracy_image",
            "eval_cls_acc_patient",
            "eval_cls_f1_patient",
            "eval_cls_roc_auc_patient",
            "eval_cls_pr_auc_patient",
            "eval_cls_acc_image",
            "eval_cls_f1_image",
            "eval_cls_roc_auc_image",
            "eval_cls_pr_auc_image",
        ]
        parts = []
        for k in keys:
            if k in logs:
                v = logs[k]
                if isinstance(v, (float, int)):
                    parts.append(f"{k}: {v:.4f}")
                else:
                    parts.append(f"{k}: {v}")
        if parts:
            text = " | ".join(parts)
            print(f"Epoch {epoch:.1f} -> {text}")
        return control


class EpochTimeCallback(TrainerCallback):
    def __init__(self, tag):
        self.tag = str(tag)
        self.epoch_start_time = None
        self.epoch_times = []

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start_time = time.time()
        return control

    def on_epoch_end(self, args, state, control, **kwargs):
        if self.epoch_start_time is not None:
            duration = time.time() - self.epoch_start_time
            self.epoch_times.append(
                {
                    "tag": self.tag,
                    "epoch": float(state.epoch) if state.epoch is not None else None,
                    "epoch_time": float(duration),
                }
            )
        return control


class RobustEarlyStoppingCallback(TrainerCallback):
    def __init__(
        self,
        metric_key="eval_mDice_fg_patient",
        patience=10,
        min_delta=0.0,
        smooth_alpha=0.6,
        overfit_patience=3,
        train_delta=0.0,
        val_delta=0.0,
        train_loss_floor=None,
        train_loss_floor_min_epoch=3,
    ):
        self.metric_key = str(metric_key)
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.smooth_alpha = float(smooth_alpha)
        self.overfit_patience = int(overfit_patience)
        self.train_delta = float(train_delta)
        self.val_delta = float(val_delta)
        self.train_loss_floor = None if train_loss_floor is None else float(train_loss_floor)
        self.train_loss_floor_min_epoch = int(train_loss_floor_min_epoch)

        self.best_ema = None
        self.bad_epochs = 0
        self.ema_metric = None
        self.ema_val_loss = None

        self.overfit_epochs = 0
        self.epoch_train_losses = {}
        self.history = []
        self.floor_triggered = False
        self.floor_triggered_epoch = None
        self.min_train_loss_seen = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.best_ema = None
        self.bad_epochs = 0
        self.ema_metric = None
        self.ema_val_loss = None
        self.overfit_epochs = 0
        self.epoch_train_losses = {}
        self.history = []
        self.floor_triggered = False
        self.floor_triggered_epoch = None
        self.min_train_loss_seen = None
        return control

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return control
        if "loss" in logs and "epoch" in logs:
            epoch = logs["epoch"]
            try:
                epoch_int = int(round(float(epoch)))
            except Exception:
                epoch_int = int(epoch)
            loss_v = float(logs["loss"])
            self.epoch_train_losses[epoch_int] = loss_v
            if self.min_train_loss_seen is None or loss_v < self.min_train_loss_seen:
                self.min_train_loss_seen = loss_v
            if self.train_loss_floor is not None and epoch_int >= self.train_loss_floor_min_epoch:
                if loss_v < self.train_loss_floor:
                    self.floor_triggered = True
                    self.floor_triggered_epoch = epoch_int
                    control.should_training_stop = True
        return control

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return control
        if "eval_loss" not in metrics or "epoch" not in metrics:
            return control

        epoch = metrics["epoch"]
        try:
            epoch_int = int(round(float(epoch)))
        except Exception:
            epoch_int = int(epoch)

        train_loss = self.epoch_train_losses.get(epoch_int)
        if train_loss is None:
            return control

        val_loss = float(metrics["eval_loss"])
        cur_metric = float(metrics.get(self.metric_key, 0.0))

        if self.ema_metric is None:
            self.ema_metric = cur_metric
        else:
            self.ema_metric = self.smooth_alpha * cur_metric + (1.0 - self.smooth_alpha) * self.ema_metric

        if self.ema_val_loss is None:
            self.ema_val_loss = val_loss
        else:
            self.ema_val_loss = self.smooth_alpha * val_loss + (1.0 - self.smooth_alpha) * self.ema_val_loss

        self.history.append(
            {
                "epoch": epoch_int,
                "train_loss": float(train_loss),
                "val_loss": float(val_loss),
                "ema_metric": float(self.ema_metric),
                "ema_val_loss": float(self.ema_val_loss),
            }
        )

        if self.best_ema is None or self.ema_metric > self.best_ema + self.min_delta:
            self.best_ema = float(self.ema_metric)
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1

        if len(self.history) >= 2:
            prev = self.history[-2]
            prev_train = float(prev["train_loss"])
            prev_val = float(prev["val_loss"])
            if float(train_loss) < prev_train - self.train_delta and float(val_loss) > prev_val + self.val_delta:
                self.overfit_epochs += 1
            else:
                self.overfit_epochs = 0

        stop_for_patience = self.bad_epochs >= self.patience
        stop_for_overfit = self.overfit_epochs >= self.overfit_patience

        if stop_for_patience or stop_for_overfit:
            control.should_training_stop = True

        return control


class EncoderUnfreezeCallback(TrainerCallback):
    def __init__(self, unfreeze_epoch=5):
        self.unfreeze_epoch = int(unfreeze_epoch)
        self.done = False

    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        if self.done:
            return control
        ep = state.epoch
        if ep is None:
            return control
        if float(ep) >= float(self.unfreeze_epoch):
            if model is not None and hasattr(model, "set_encoder_trainable"):
                model.set_encoder_trainable(True)
            self.done = True
        return control


def save_log_history(log_history, out_path):
    if not log_history:
        return
    keys = set()
    for entry in log_history:
        keys.update(entry.keys())
    fieldnames = sorted(keys)
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for entry in log_history:
            row = {}
            for k in fieldnames:
                v = entry.get(k)
                if isinstance(v, (int, float, str)) or v is None:
                    row[k] = v
                else:
                    row[k] = str(v)
            writer.writerow(row)


def save_epoch_times(epoch_times, out_path):
    if not epoch_times:
        return
    fieldnames = sorted(epoch_times[0].keys())
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in epoch_times:
            writer.writerow(row)


def append_log_history_to_global_csv(log_history, out_path, extra_info=None):
    if not log_history:
        return
    keys = set()
    for entry in log_history:
        keys.update(entry.keys())
    if extra_info is not None:
        keys.update(extra_info.keys())
    fieldnames = sorted(keys)
    file_exists = os.path.exists(out_path)
    with open(out_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        for entry in log_history:
            row = {}
            for k in fieldnames:
                if extra_info is not None and k in extra_info:
                    v = extra_info[k]
                else:
                    v = entry.get(k)
                if isinstance(v, (int, float, str)) or v is None:
                    row[k] = v
                else:
                    row[k] = str(v)
            writer.writerow(row)


class MultiTaskSegClsModel(nn.Module):
    def __init__(
        self,
        checkpoint,
        id2label,
        label2id,
        class_weights=None,
        focal_gamma=None,
        cls_loss_weight=1.0,
        cls_class_weights=None,
        pool_mode="mask",
        pool_alpha=1.0,
        freeze_encoder=False,
    ):
        super().__init__()
        self.seg_model = SegformerForSemanticSegmentation.from_pretrained(
            checkpoint,
            num_labels=NUM_SEG_LABELS,
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True,
        )
        hidden_dim = int(self.seg_model.config.hidden_sizes[-1])
        self.cls_classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, NUM_CLS_LABELS),
        )

        self.pool_mode = str(pool_mode)
        self.pool_alpha = float(pool_alpha)

        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        else:
            self.class_weights = None

        if cls_class_weights is not None:
            self.cls_class_weights = torch.tensor(cls_class_weights, dtype=torch.float32)
        else:
            self.cls_class_weights = None

        self.focal_gamma = focal_gamma
        self.cls_loss_weight = float(cls_loss_weight)
        self._freeze_encoder_initial = bool(freeze_encoder)

    def set_encoder_trainable(self, trainable):
        if hasattr(self.seg_model, "segformer"):
            for p in self.seg_model.segformer.parameters():
                p.requires_grad = bool(trainable)

    def freeze_encoder_if_needed(self):
        if self._freeze_encoder_initial:
            self.set_encoder_trainable(False)

    def forward(self, pixel_values=None, labels=None, cls_labels=None):
        outputs_seg = self.seg_model(pixel_values=pixel_values, output_hidden_states=True, return_dict=True)
        seg_logits = outputs_seg.logits
        last_feat = outputs_seg.hidden_states[-1]

        pooled_global = F.adaptive_avg_pool2d(last_feat, output_size=1).view(last_feat.size(0), -1)

        if self.pool_mode == "mask":
            probs = torch.softmax(seg_logits, dim=1)[:, 1:2]
            mask_ds = F.interpolate(probs, size=last_feat.shape[-2:], mode="bilinear", align_corners=False)
            denom = mask_ds.sum(dim=(2, 3)).clamp_min(1e-6)
            pooled_mask = (last_feat * mask_ds).sum(dim=(2, 3)) / denom
            pooled = self.pool_alpha * pooled_mask + (1.0 - self.pool_alpha) * pooled_global
        else:
            pooled = pooled_global

        cls_logits = self.cls_classifier(pooled)

        loss = None
        if labels is not None:
            loss_seg = segmentation_loss(seg_logits, labels, self.class_weights, self.focal_gamma)
            loss = loss_seg
        if cls_labels is not None:
            cls_labels = cls_labels.to(cls_logits.device)
            w = self.cls_class_weights.to(cls_logits.device) if self.cls_class_weights is not None else None
            loss_cls = F.cross_entropy(cls_logits, cls_labels, weight=w, ignore_index=-100)
            loss = (self.cls_loss_weight * loss_cls) if loss is None else (loss + self.cls_loss_weight * loss_cls)

        if loss is None:
            return {"loss": None, "logits": seg_logits, "cls_logits": cls_logits}
        return {"loss": loss, "logits": seg_logits, "cls_logits": cls_logits}


class MultiTaskTrainer(Trainer):
    def __init__(
        self,
        *args,
        use_patient_balanced_sampling=False,
        patient_balanced_seed=0,
        freeze_encoder=False,
        unfreeze_epoch=5,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.use_patient_balanced_sampling = bool(use_patient_balanced_sampling)
        self.patient_balanced_seed = int(patient_balanced_seed)
        self.freeze_encoder = bool(freeze_encoder)
        self.unfreeze_epoch = int(unfreeze_epoch)
        self._encoder_frozen_applied = False

    def _model_inputs_only(self, inputs):
        allowed = ("pixel_values", "labels", "cls_labels")
        return {k: v for k, v in inputs.items() if k in allowed}

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        model_inputs = self._model_inputs_only(inputs)
        outputs = model(**model_inputs)
        loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss
        if return_outputs:
            return loss, outputs
        return loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)

        seg_labels = inputs.get("labels", None)
        cls_labels = inputs.get("cls_labels", None)
        patient_idx = inputs.get("patient_idx", None)

        model_inputs = self._model_inputs_only(inputs)

        with torch.no_grad():
            outputs = model(**model_inputs)

        if isinstance(outputs, dict):
            loss = outputs.get("loss", None)
            seg_logits = outputs.get("logits", None)
            cls_logits = outputs.get("cls_logits", None)
        else:
            loss = getattr(outputs, "loss", None)
            seg_logits = getattr(outputs, "logits", None)
            cls_logits = None

        if prediction_loss_only:
            return (loss, None, None)

        preds = (seg_logits, cls_logits)
        labs = (seg_labels, cls_labels, patient_idx)
        return (loss, preds, labs)

    def create_optimizer_and_scheduler(self, num_training_steps):
        super().create_optimizer_and_scheduler(num_training_steps)
        if self.freeze_encoder and (not self._encoder_frozen_applied):
            if hasattr(self.model, "freeze_encoder_if_needed"):
                self.model.freeze_encoder_if_needed()
            self._encoder_frozen_applied = True

    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")
        if self.use_patient_balanced_sampling:
            if hasattr(self.train_dataset, "image_paths"):
                sampler = make_patient_balanced_sampler(self.train_dataset.image_paths, self.patient_balanced_seed)
                return DataLoader(
                    self.train_dataset,
                    batch_size=self.args.train_batch_size,
                    sampler=sampler,
                    collate_fn=self.data_collator,
                    num_workers=self.args.dataloader_num_workers,
                    pin_memory=self.args.dataloader_pin_memory,
                    drop_last=self.args.dataloader_drop_last,
                )
        return super().get_train_dataloader()


def plot_confusion(cm, classes, title, save_path):
    fig, ax = plt.subplots()
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(str(title))
    fig.colorbar(im)
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(int(cm[i, j]), "d"), ha="center", va="center")
    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")
    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)


def plot_roc_curve(fpr, tpr, roc_auc, title, save_path):
    fig, ax = plt.subplots()
    ax.plot(fpr, tpr, label=f"AUC = {float(roc_auc):.3f}")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(str(title))
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)


def plot_pr_curve(precision, recall, pr_auc, title, save_path):
    fig, ax = plt.subplots()
    ax.plot(recall, precision, label=f"AUC = {float(pr_auc):.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(str(title))
    ax.legend(loc="lower left")
    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)


def run_inference_on_test(model, processor, image_paths, output_root, subdir_name="test_predictions"):
    if len(image_paths) == 0:
        return

    device = next(model.parameters()).device
    model.eval()

    pred_dir = os.path.join(output_root, subdir_name)
    os.makedirs(pred_dir, exist_ok=True)
    csv_path = os.path.join(pred_dir, "predictions.csv")

    rows = []

    seg_cm = np.zeros((NUM_SEG_LABELS, NUM_SEG_LABELS), dtype=np.int64)
    seg_intersections = np.zeros(NUM_SEG_LABELS, dtype=np.float64)
    seg_unions = np.zeros(NUM_SEG_LABELS, dtype=np.float64)
    seg_denoms = np.zeros(NUM_SEG_LABELS, dtype=np.float64)

    patient_seg_stats = {}
    patient_cls_scores = {}

    y_true_img = []
    y_pred_img = []
    y_score_img = []

    t_start = time.time()

    for image_path in image_paths:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)

        with torch.no_grad():
            outputs = model(pixel_values=pixel_values, labels=None, cls_labels=None)

        if isinstance(outputs, dict):
            seg_logits = outputs["logits"]
            cls_logits = outputs["cls_logits"]
        else:
            seg_logits = outputs.logits
            cls_logits = None

        h, w = int(image.size[1]), int(image.size[0])
        upsampled_logits = F.interpolate(seg_logits, size=(h, w), mode="bilinear", align_corners=False)
        pred_mask = upsampled_logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)

        gt_mask = load_mask_array(image_path).astype(np.uint8)

        for cid in range(NUM_SEG_LABELS):
            pred_c = pred_mask == cid
            gt_c = gt_mask == cid
            inter = np.logical_and(pred_c, gt_c).sum()
            uni = np.logical_or(pred_c, gt_c).sum()
            denom = pred_c.sum() + gt_c.sum()
            seg_intersections[cid] += inter
            seg_unions[cid] += uni
            seg_denoms[cid] += denom

        gt_flat = gt_mask.reshape(-1).astype(np.int64)
        pred_flat = pred_mask.reshape(-1).astype(np.int64)
        idx = gt_flat * NUM_SEG_LABELS + pred_flat
        cm_img = np.bincount(idx, minlength=NUM_SEG_LABELS * NUM_SEG_LABELS).reshape(NUM_SEG_LABELS, NUM_SEG_LABELS)
        seg_cm += cm_img

        positive_pixels = int((pred_mask == 1).sum())
        total_pixels = int(pred_mask.size)
        positive_ratio = positive_pixels / float(total_pixels) if total_pixels > 0 else 0.0

        cls_head_class = "unknown"
        cls_head_prob = 0.0
        arthrit_prob = None

        if cls_logits is not None:
            probs_cls = torch.softmax(cls_logits, dim=1)[0].detach().cpu().numpy()
            cls_pred_idx = int(np.argmax(probs_cls))
            arthrit_prob = float(probs_cls[1])
            cls_head_prob = float(probs_cls[cls_pred_idx])
            cls_head_class = "arthrit" if cls_pred_idx == 1 else "control"

        img_np = np.array(image)
        mask_uint8 = ((pred_mask == 1).astype(np.uint8)) * 255
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        overlay = img_np.copy()
        if len(contours) > 0:
            cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)
        blended = (0.6 * img_np + 0.4 * overlay).astype(np.uint8)
        out_img = Image.fromarray(blended)

        base = os.path.basename(image_path)
        pred_label = cls_head_class if cls_head_class in ("control", "arthrit") else "unknown"
        out_name = os.path.splitext(base)[0] + f"_pred_{pred_label}.png"
        out_img.save(os.path.join(pred_dir, out_name))

        pid = get_patient_id_from_path(image_path)
        true_group = get_patient_group_from_id(pid)
        patient_base = get_patient_base_id(pid)

        if patient_base not in patient_seg_stats:
            patient_seg_stats[patient_base] = {
                "inter": np.zeros(NUM_SEG_LABELS, dtype=np.float64),
                "union": np.zeros(NUM_SEG_LABELS, dtype=np.float64),
                "denom": np.zeros(NUM_SEG_LABELS, dtype=np.float64),
            }
        for cid in range(NUM_SEG_LABELS):
            pred_c = pred_mask == cid
            gt_c = gt_mask == cid
            patient_seg_stats[patient_base]["inter"][cid] += float(np.logical_and(pred_c, gt_c).sum())
            patient_seg_stats[patient_base]["union"][cid] += float(np.logical_or(pred_c, gt_c).sum())
            patient_seg_stats[patient_base]["denom"][cid] += float(pred_c.sum() + gt_c.sum())

        if (
            true_group in ("control", "arthrit")
            and cls_head_class in ("control", "arthrit")
            and arthrit_prob is not None
        ):
            label_int = 1 if true_group == "arthrit" else 0
            pred_int = 1 if cls_head_class == "arthrit" else 0
            y_true_img.append(label_int)
            y_pred_img.append(pred_int)
            y_score_img.append(float(arthrit_prob))

            if patient_base not in patient_cls_scores:
                patient_cls_scores[patient_base] = {"true": label_int, "scores": []}
            patient_cls_scores[patient_base]["scores"].append(float(arthrit_prob))

        rows.append(
            [
                image_path,
                pid,
                total_pixels,
                positive_pixels,
                positive_ratio,
                true_group,
                cls_head_class,
                float(cls_head_prob),
                float(arthrit_prob) if arthrit_prob is not None else 0.0,
            ]
        )

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(
            [
                "image_path",
                "patient_id",
                "total_pixels",
                "positive_pixels",
                "positive_ratio",
                "true_group",
                "cls_head_class",
                "cls_head_prob",
                "arthrit_prob",
            ]
        )
        writer.writerows(rows)

    def safe_div(a, b):
        return a / b if b > 0 else 0.0

    seg_iou = []
    seg_dice = []
    for cid in range(NUM_SEG_LABELS):
        inter = float(seg_intersections[cid])
        uni = float(seg_unions[cid])
        denom = float(seg_denoms[cid])
        iou_c = float(inter / uni) if uni > 0 else 0.0
        dice_c = float(2.0 * inter / denom) if denom > 0 else 0.0
        seg_iou.append(iou_c)
        seg_dice.append(dice_c)

    mean_seg_iou = float(np.mean(seg_iou)) if len(seg_iou) > 0 else 0.0
    mean_seg_dice = float(np.mean(seg_dice)) if len(seg_dice) > 0 else 0.0

    plot_confusion(seg_cm, ["background", "fluid"], f"{subdir_name} segmentation", os.path.join(pred_dir, "confusion_seg.png"))

    patient_miou = []
    patient_mdice = []
    patient_miou_fg = []
    patient_mdice_fg = []

    for pb, st in patient_seg_stats.items():
        inter = st["inter"]
        union = st["union"]
        denom = st["denom"]
        iou_c = np.where(union > 0, inter / union, 0.0)
        dice_c = np.where(denom > 0, (2.0 * inter) / denom, 0.0)
        patient_miou.append(float(iou_c.mean()))
        patient_mdice.append(float(dice_c.mean()))
        if NUM_SEG_LABELS > 1:
            patient_miou_fg.append(float(iou_c[1:].mean()))
            patient_mdice_fg.append(float(dice_c[1:].mean()))
        else:
            patient_miou_fg.append(float(iou_c[0]))
            patient_mdice_fg.append(float(dice_c[0]))

    mean_seg_iou_patient = float(np.mean(patient_miou)) if len(patient_miou) else 0.0
    mean_seg_dice_patient = float(np.mean(patient_mdice)) if len(patient_mdice) else 0.0
    mean_seg_iou_fg_patient = float(np.mean(patient_miou_fg)) if len(patient_miou_fg) else 0.0
    mean_seg_dice_fg_patient = float(np.mean(patient_mdice_fg)) if len(patient_mdice_fg) else 0.0

    roc_auc_img, pr_auc_img = _safe_auc(np.array(y_true_img, dtype=np.int64), np.array(y_score_img, dtype=np.float64))
    f1_img = float(f1_score(np.array(y_true_img), np.array(y_pred_img))) if len(y_true_img) else 0.0

    p_true = []
    p_score = []
    for pb, info in patient_cls_scores.items():
        if len(info["scores"]) == 0:
            continue
        sc = float(np.mean(info["scores"]))
        if not np.isfinite(sc):
            continue
        p_true.append(int(info["true"]))
        p_score.append(sc)

    if len(p_true) > 0:
        p_true_np = np.array(p_true, dtype=np.int64)
        p_score_np = np.array(p_score, dtype=np.float64)
        p_pred_np = (p_score_np >= 0.5).astype(np.int64)
        acc_patient = float((p_pred_np == p_true_np).mean())
        try:
            f1_patient = float(f1_score(p_true_np, p_pred_np))
        except Exception:
            f1_patient = 0.0
        roc_auc_patient, pr_auc_patient = _safe_auc(p_true_np, p_score_np)
        cls_cm = confusion_matrix(p_true_np, p_pred_np)
        plot_confusion(cls_cm, ["control", "arthrit"], f"{subdir_name} classification patient", os.path.join(pred_dir, "confusion_cls_patient.png"))
        if len(np.unique(p_true_np)) > 1:
            fpr_c, tpr_c, _ = roc_curve(p_true_np, p_score_np)
            plot_roc_curve(fpr_c, tpr_c, roc_auc_patient, f"{subdir_name} patient ROC", os.path.join(pred_dir, "roc_cls_patient.png"))
            prec_c, rec_c, _ = precision_recall_curve(p_true_np, p_score_np)
            plot_pr_curve(prec_c, rec_c, pr_auc_patient, f"{subdir_name} patient PR", os.path.join(pred_dir, "pr_cls_patient.png"))
    else:
        acc_patient = 0.0
        f1_patient = 0.0
        roc_auc_patient = 0.0
        pr_auc_patient = 0.0

    print(
        f"[{subdir_name}] Segmentation (image-pixel pooled): mIoU={mean_seg_iou:.3f}, mDice={mean_seg_dice:.3f}, IoU0={seg_iou[0]:.3f}, IoU1={seg_iou[1]:.3f}, Dice0={seg_dice[0]:.3f}, Dice1={seg_dice[1]:.3f}"
    )
    print(
        f"[{subdir_name}] Segmentation (patient-avg): mIoU_patient={mean_seg_iou_patient:.3f}, mDice_patient={mean_seg_dice_patient:.3f}, mIoU_fg_patient={mean_seg_iou_fg_patient:.3f}, mDice_fg_patient={mean_seg_dice_fg_patient:.3f}"
    )
    if len(y_true_img) > 0:
        print(f"[{subdir_name}] Classification (image): F1={f1_img:.3f}, ROC AUC={roc_auc_img:.3f}, PR AUC={pr_auc_img:.3f}")
    print(f"[{subdir_name}] Classification (patient): acc={acc_patient:.3f}, F1={f1_patient:.3f}, ROC AUC={roc_auc_patient:.3f}, PR AUC={pr_auc_patient:.3f}")

    total_inference_time = float(time.time() - t_start)
    inference_time_per_image = total_inference_time / float(len(image_paths)) if len(image_paths) > 0 else 0.0

    metrics_all = {
        "segmentation": {
            "image_pixel_pooled": {
                "mIoU": float(mean_seg_iou),
                "mDice": float(mean_seg_dice),
                "iou_class_0": float(seg_iou[0]) if len(seg_iou) > 0 else 0.0,
                "iou_class_1": float(seg_iou[1]) if len(seg_iou) > 1 else 0.0,
                "dice_class_0": float(seg_dice[0]) if len(seg_dice) > 0 else 0.0,
                "dice_class_1": float(seg_dice[1]) if len(seg_dice) > 1 else 0.0,
            },
            "patient_avg": {
                "mIoU_patient": float(mean_seg_iou_patient),
                "mDice_patient": float(mean_seg_dice_patient),
                "mIoU_fg_patient": float(mean_seg_iou_fg_patient),
                "mDice_fg_patient": float(mean_seg_dice_fg_patient),
            },
        },
        "classification": {
            "image": {
                "n_images": int(len(y_true_img)),
                "f1": float(f1_img),
                "roc_auc": float(roc_auc_img),
                "pr_auc": float(pr_auc_img),
            },
            "patient": {
                "n_patients": int(len(p_true)),
                "acc": float(acc_patient),
                "f1": float(f1_patient),
                "roc_auc": float(roc_auc_patient),
                "pr_auc": float(pr_auc_patient),
            },
        },
        "subdir_name": str(subdir_name),
        "inference_time_seconds": float(total_inference_time),
        "inference_time_per_image_seconds": float(inference_time_per_image),
    }

    metrics_path = os.path.join(pred_dir, "metrics_summary.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics_all, f, indent=2)


def make_training_args(**kwargs):
    try:
        return TrainingArguments(**kwargs)
    except TypeError:
        if "evaluation_strategy" in kwargs and "eval_strategy" not in kwargs:
            kwargs2 = dict(kwargs)
            kwargs2["eval_strategy"] = kwargs2.pop("evaluation_strategy")
            try:
                return TrainingArguments(**kwargs2)
            except TypeError:
                kwargs = kwargs2
        if "lr_scheduler_type" in kwargs:
            kwargs2 = dict(kwargs)
            kwargs2.pop("lr_scheduler_type", None)
            return TrainingArguments(**kwargs2)
        raise


def train_one_config(
    train_paths,
    val_paths,
    run_output_dir,
    num_epochs,
    train_batch_size,
    eval_batch_size,
    learning_rate,
    weight_decay,
    seed,
    checkpoint,
    id2label,
    label2id,
    focal_gamma,
    cls_loss_weight,
    control_boost,
    arthrit_boost,
    early_patience=10,
    early_min_delta=0.0,
    early_smooth_alpha=0.6,
    overfit_patience=3,
    train_delta=0.0,
    val_delta=0.0,
    global_epoch_log_path=None,
    global_extra_info=None,
    pool_mode="mask",
    pool_alpha=1.0,
    train_loss_floor=None,
    train_loss_floor_min_epoch=3,
    fp16_if_available=True,
    use_patient_balanced_sampling=True,
    freeze_encoder=True,
    unfreeze_epoch=2,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
):
    set_seed(int(seed))
    os.makedirs(run_output_dir, exist_ok=True)

    image_processor = AutoImageProcessor.from_pretrained(checkpoint, do_reduce_labels=False)
    seg_class_weights = compute_seg_class_weights(train_paths)
    cls_class_weights = compute_cls_class_weights(
        train_paths,
        control_boost=float(control_boost),
        arthrit_boost=float(arthrit_boost),
    )

    model = MultiTaskSegClsModel(
        checkpoint=checkpoint,
        id2label=id2label,
        label2id=label2id,
        class_weights=seg_class_weights,
        focal_gamma=focal_gamma,
        cls_loss_weight=float(cls_loss_weight),
        cls_class_weights=cls_class_weights,
        pool_mode=str(pool_mode),
        pool_alpha=float(pool_alpha),
        freeze_encoder=bool(freeze_encoder),
    )

    train_dataset = USGSegmentationClassificationDataset(train_paths, image_processor, include_cls_labels=True)
    val_dataset = USGSegmentationClassificationDataset(val_paths, image_processor, include_cls_labels=True)

    use_fp16 = bool(fp16_if_available) and bool(torch.cuda.is_available())

    training_args = make_training_args(
        output_dir=run_output_dir,
        learning_rate=float(learning_rate),
        num_train_epochs=int(num_epochs),
        per_device_train_batch_size=int(train_batch_size),
        per_device_eval_batch_size=int(eval_batch_size),
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_mDice_fg_patient",
        greater_is_better=True,
        logging_dir=os.path.join(run_output_dir, "logs"),
        logging_strategy="epoch",
        seed=int(seed),
        dataloader_num_workers=0,
        fp16=bool(use_fp16),
        weight_decay=float(weight_decay),
        remove_unused_columns=False,
        report_to="none",
        disable_tqdm=True,
        warmup_ratio=0.1,
        lr_scheduler_type=str(lr_scheduler_type),
        max_grad_norm=float(max_grad_norm),
    )

    epoch_time_cb = EpochTimeCallback(tag="run")
    es_cb = RobustEarlyStoppingCallback(
        metric_key="eval_mDice_fg_patient",
        patience=int(early_patience),
        min_delta=float(early_min_delta),
        smooth_alpha=float(early_smooth_alpha),
        overfit_patience=int(overfit_patience),
        train_delta=float(train_delta),
        val_delta=float(val_delta),
        train_loss_floor=(None if train_loss_floor is None else float(train_loss_floor)),
        train_loss_floor_min_epoch=int(train_loss_floor_min_epoch),
    )

    callbacks = [
        EpochMetricsPrinter(),
        es_cb,
        epoch_time_cb,
        EncoderUnfreezeCallback(unfreeze_epoch=int(unfreeze_epoch)),
    ]

    trainer = MultiTaskTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
        use_patient_balanced_sampling=bool(use_patient_balanced_sampling),
        patient_balanced_seed=int(seed),
        freeze_encoder=bool(freeze_encoder),
        unfreeze_epoch=int(unfreeze_epoch),
    )

    train_result = trainer.train()

    history_path = os.path.join(run_output_dir, "training_history.csv")
    save_log_history(trainer.state.log_history, history_path)

    if global_epoch_log_path is not None:
        append_log_history_to_global_csv(trainer.state.log_history, global_epoch_log_path, global_extra_info)

    epoch_times_path = os.path.join(run_output_dir, "epoch_times.csv")
    save_epoch_times(epoch_time_cb.epoch_times, epoch_times_path)

    metrics_val = trainer.evaluate(val_dataset)
    metrics_all = {}
    metrics_all.update(train_result.metrics)
    metrics_all.update(metrics_val)
    with open(os.path.join(run_output_dir, "metrics_summary.json"), "w") as f:
        json.dump(metrics_all, f, indent=2, default=float)

    mdice_patient = float(metrics_val.get("eval_mDice_fg_patient", 0.0))
    train_runtime = float(train_result.metrics.get("train_runtime", 0.0))
    min_train_loss_seen = es_cb.min_train_loss_seen if es_cb.min_train_loss_seen is not None else None
    floor_triggered = bool(es_cb.floor_triggered)

    return mdice_patient, train_runtime, floor_triggered, min_train_loss_seen


def select_best_hyperparams(
    train_val_usg_dirs,
    search_root,
    val_ratio,
    base_seed,
    search_epochs,
    checkpoint,
    id2label,
    label2id,
    k_inner_splits=3,
    early_patience=10,
    early_min_delta=0.0,
    early_smooth_alpha=0.6,
    overfit_patience=3,
    train_delta=0.0,
    val_delta=0.0,
    lr_min=3e-5,
    lr_max=1e-4,
    wd_min=0.0,
    wd_max=0.01,
    search_n_trials=20,
    pool_mode="mask",
    pool_alpha=1.0,
    train_loss_floor=0.01,
    train_loss_floor_min_epoch=3,
    use_patient_balanced_sampling=True,
    freeze_encoder=True,
    unfreeze_epoch=2,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
):
    os.makedirs(search_root, exist_ok=True)
    global_search_epoch_log = os.path.join(search_root, "all_search_epochs_log.csv")
    cfg_summaries = []

    def objective(trial):
        lr = trial.suggest_float("learning_rate", float(lr_min), float(lr_max), log=True)
        wd = trial.suggest_float("weight_decay", float(wd_min), float(wd_max))
        train_bs_trial = trial.suggest_categorical("train_batch_size", [1, 2, 4])
        eval_bs_trial = trial.suggest_categorical("eval_batch_size", [1, 2, 4])
        cls_loss_weight_trial = trial.suggest_float("cls_loss_weight", 0.25, 4.0, log=True)
        use_focal = trial.suggest_categorical("use_focal_loss", [False, True])
        focal_gamma_trial = None
        if use_focal:
            focal_gamma_trial = trial.suggest_float("focal_gamma", 0.5, 3.0)
        control_boost_trial = trial.suggest_float("control_boost", 1.0, 3.0)
        arthrit_boost_trial = trial.suggest_float("arthrit_boost", 0.5, 2.0)

        fold_scores = []
        fold_runtimes = []
        fold_floor_hits = 0

        for fold_idx, train_val_usg_dir in enumerate(train_val_usg_dirs):
            split_scores = []
            split_runtimes = []

            for split_idx in range(int(k_inner_splits)):
                split_seed = int(base_seed) + int(fold_idx) * 10000 + int(split_idx)
                train_paths, val_paths = split_patients(train_val_usg_dir, float(val_ratio), int(split_seed))

                run_output_dir = os.path.join(
                    search_root, f"trial_{trial.number}_fold_{fold_idx + 1}_split_{split_idx + 1}"
                )

                extra_info = {
                    "trial": int(trial.number),
                    "fold_idx": int(fold_idx + 1),
                    "split_idx": int(split_idx + 1),
                    "learning_rate": float(lr),
                    "weight_decay": float(wd),
                    "train_batch_size": int(train_bs_trial),
                    "eval_batch_size": int(eval_bs_trial),
                    "cls_loss_weight": float(cls_loss_weight_trial),
                    "use_focal_loss": bool(use_focal),
                    "focal_gamma": float(focal_gamma_trial) if focal_gamma_trial is not None else None,
                    "control_boost": float(control_boost_trial),
                    "arthrit_boost": float(arthrit_boost_trial),
                    "seed": int(split_seed),
                    "pool_mode": str(pool_mode),
                    "pool_alpha": float(pool_alpha),
                    "patient_balanced_sampling": bool(use_patient_balanced_sampling),
                    "freeze_encoder": bool(freeze_encoder),
                    "unfreeze_epoch": int(unfreeze_epoch),
                    "lr_scheduler_type": str(lr_scheduler_type),
                    "max_grad_norm": float(max_grad_norm),
                }

                mdice_patient, train_runtime, floor_triggered, min_train_loss_seen = train_one_config(
                    train_paths=train_paths,
                    val_paths=val_paths,
                    run_output_dir=run_output_dir,
                    num_epochs=int(search_epochs),
                    train_batch_size=int(train_bs_trial),
                    eval_batch_size=int(eval_bs_trial),
                    learning_rate=float(lr),
                    weight_decay=float(wd),
                    seed=int(split_seed),
                    checkpoint=checkpoint,
                    id2label=id2label,
                    label2id=label2id,
                    focal_gamma=focal_gamma_trial,
                    cls_loss_weight=float(cls_loss_weight_trial),
                    control_boost=float(control_boost_trial),
                    arthrit_boost=float(arthrit_boost_trial),
                    early_patience=int(early_patience),
                    early_min_delta=float(early_min_delta),
                    early_smooth_alpha=float(early_smooth_alpha),
                    overfit_patience=int(overfit_patience),
                    train_delta=float(train_delta),
                    val_delta=float(val_delta),
                    global_epoch_log_path=global_search_epoch_log,
                    global_extra_info=extra_info,
                    pool_mode=str(pool_mode),
                    pool_alpha=float(pool_alpha),
                    train_loss_floor=(None if train_loss_floor is None else float(train_loss_floor)),
                    train_loss_floor_min_epoch=int(train_loss_floor_min_epoch),
                    fp16_if_available=True,
                    use_patient_balanced_sampling=bool(use_patient_balanced_sampling),
                    freeze_encoder=bool(freeze_encoder),
                    unfreeze_epoch=int(unfreeze_epoch),
                    lr_scheduler_type=str(lr_scheduler_type),
                    max_grad_norm=float(max_grad_norm),
                )

                if floor_triggered:
                    fold_floor_hits += 1

                cfg_summaries.append(
                    {
                        "trial": int(trial.number),
                        "fold_idx": int(fold_idx + 1),
                        "split_idx": int(split_idx + 1),
                        "learning_rate": float(lr),
                        "weight_decay": float(wd),
                        "train_batch_size": int(train_bs_trial),
                        "eval_batch_size": int(eval_bs_trial),
                        "cls_loss_weight": float(cls_loss_weight_trial),
                        "use_focal_loss": bool(use_focal),
                        "focal_gamma": float(focal_gamma_trial) if focal_gamma_trial is not None else None,
                        "control_boost": float(control_boost_trial),
                        "arthrit_boost": float(arthrit_boost_trial),
                        "pool_mode": str(pool_mode),
                        "pool_alpha": float(pool_alpha),
                        "seed": int(split_seed),
                        "val_mDice_fg_patient": float(mdice_patient),
                        "train_runtime": float(train_runtime),
                        "min_train_loss_seen": float(min_train_loss_seen) if min_train_loss_seen is not None else None,
                        "floor_triggered": bool(floor_triggered),
                        "patient_balanced_sampling": bool(use_patient_balanced_sampling),
                        "freeze_encoder": bool(freeze_encoder),
                        "unfreeze_epoch": int(unfreeze_epoch),
                        "lr_scheduler_type": str(lr_scheduler_type),
                        "max_grad_norm": float(max_grad_norm),
                    }
                )

                split_scores.append(float(mdice_patient))
                split_runtimes.append(float(train_runtime))

                step = int(fold_idx) * int(k_inner_splits) + int(split_idx) + 1
                trial.report(float(np.median(split_scores)), step=step)
                if trial.should_prune():
                    raise optuna.TrialPruned()

            fold_score = float(np.median(split_scores)) if len(split_scores) > 0 else 0.0
            fold_runtime = float(np.mean(split_runtimes)) if len(split_runtimes) > 0 else 0.0
            fold_scores.append(float(fold_score))
            fold_runtimes.append(float(fold_runtime))

        if train_loss_floor is not None and fold_floor_hits > 0:
            raise optuna.TrialPruned()

        score = float(np.median(fold_scores)) if len(fold_scores) > 0 else 0.0
        mean_runtime = float(np.mean(fold_runtimes)) if len(fold_runtimes) > 0 else 0.0
        trial.set_user_attr("mean_train_runtime", float(mean_runtime))
        return score

    pruner = optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=0, interval_steps=1)
    study = optuna.create_study(direction="maximize", pruner=pruner)
    study.optimize(objective, n_trials=int(search_n_trials))

    completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None]
    if len(completed_trials) > 0:
        best_trial = max(completed_trials, key=lambda t: float(t.value))
        best_params = best_trial.params
        best_cfg = {
            "learning_rate": float(best_params.get("learning_rate", float(np.sqrt(float(lr_min) * float(lr_max))))),
            "weight_decay": float(best_params.get("weight_decay", float((float(wd_min) + float(wd_max)) / 2.0))),
            "train_batch_size": int(best_params.get("train_batch_size", 2)),
            "eval_batch_size": int(best_params.get("eval_batch_size", 2)),
            "cls_loss_weight": float(best_params.get("cls_loss_weight", 1.0)),
            "use_focal_loss": bool(best_params.get("use_focal_loss", False)),
            "focal_gamma": float(best_params["focal_gamma"]) if "focal_gamma" in best_params else None,
            "control_boost": float(best_params.get("control_boost", 1.0)),
            "arthrit_boost": float(best_params.get("arthrit_boost", 1.0)),
            "pool_mode": str(pool_mode),
            "pool_alpha": float(pool_alpha),
            "mean_val_mDice_fg_patient": float(best_trial.value),
            "mean_train_runtime": float(best_trial.user_attrs.get("mean_train_runtime", 0.0)),
            "patient_balanced_sampling": bool(use_patient_balanced_sampling),
            "freeze_encoder": bool(freeze_encoder),
            "unfreeze_epoch": int(unfreeze_epoch),
            "lr_scheduler_type": str(lr_scheduler_type),
            "max_grad_norm": float(max_grad_norm),
        }
    else:
        best_cfg = {
            "learning_rate": float(np.sqrt(float(lr_min) * float(lr_max))),
            "weight_decay": float((float(wd_min) + float(wd_max)) / 2.0),
            "train_batch_size": 2,
            "eval_batch_size": 2,
            "cls_loss_weight": 1.0,
            "use_focal_loss": False,
            "focal_gamma": None,
            "control_boost": 1.0,
            "arthrit_boost": 1.0,
            "pool_mode": str(pool_mode),
            "pool_alpha": float(pool_alpha),
            "mean_val_mDice_fg_patient": 0.0,
            "mean_train_runtime": 0.0,
            "patient_balanced_sampling": bool(use_patient_balanced_sampling),
            "freeze_encoder": bool(freeze_encoder),
            "unfreeze_epoch": int(unfreeze_epoch),
            "lr_scheduler_type": str(lr_scheduler_type),
            "max_grad_norm": float(max_grad_norm),
        }

    if cfg_summaries:
        fieldnames = sorted(cfg_summaries[0].keys())
        csv_path = os.path.join(search_root, "hyperparam_search_summary.csv")
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for row in cfg_summaries:
                writer.writerow(row)

        history_obj = {"candidates": cfg_summaries, "best_cfg": best_cfg}
        torch.save(history_obj, os.path.join(search_root, "hyperparam_history.pt"))

    return best_cfg


def run_single_fold(
    fold_root,
    output_root,
    best_cfg,
    best_cfg_mdice_patient,
    final_epochs,
    val_ratio,
    seed,
    checkpoint,
    external_test_usg_dir=None,
    num_runs=5,
    early_patience=20,
    early_min_delta=1e-4,
    early_smooth_alpha=0.6,
    overfit_patience=3,
    train_delta=0.0,
    val_delta=0.0,
    use_patient_balanced_sampling=True,
    freeze_encoder=True,
    unfreeze_epoch=5,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    fp16_if_available=True,
):
    id2label = {0: "background", 1: "fluid"}
    label2id = {"background": 0, "fluid": 1}
    os.makedirs(output_root, exist_ok=True)

    fold_name = os.path.basename(fold_root.rstrip(os.sep))
    runs_summary = []

    train_val_usg_dir = os.path.join(fold_root, "train_val", "darkened")
    fold_test_paths = get_fold_test_image_paths(fold_root)
    external_test_paths = get_external_test_image_paths(external_test_usg_dir)

    best_lr = float(best_cfg["learning_rate"])
    best_wd = float(best_cfg["weight_decay"])
    best_train_bs = int(best_cfg.get("train_batch_size", 2))
    best_eval_bs = int(best_cfg.get("eval_batch_size", 2))
    best_cls_loss_weight = float(best_cfg.get("cls_loss_weight", 1.0))
    best_focal_gamma = best_cfg.get("focal_gamma", None)
    best_control_boost = float(best_cfg.get("control_boost", 1.0))
    best_arthrit_boost = float(best_cfg.get("arthrit_boost", 1.0))
    pool_mode = best_cfg.get("pool_mode", "mask")
    pool_alpha = float(best_cfg.get("pool_alpha", 1.0))

    fold_global_final_epoch_log = os.path.join(output_root, "all_final_epochs_log.csv")

    for run_idx in range(int(num_runs)):
        run_seed = int(seed) + int(run_idx) * 1000
        run_tag = f"run_{run_idx + 1}"
        run_output_root = os.path.join(output_root, run_tag)
        final_root = os.path.join(run_output_root, "final")
        os.makedirs(final_root, exist_ok=True)

        final_split_seed = int(run_seed) + 999
        train_paths_final, val_paths_final = split_patients(train_val_usg_dir, float(val_ratio), int(final_split_seed))

        set_seed(int(run_seed))
        image_processor = AutoImageProcessor.from_pretrained(checkpoint, do_reduce_labels=False)
        seg_class_weights_full = compute_seg_class_weights(train_paths_final)
        cls_class_weights_full = compute_cls_class_weights(
            train_paths_final,
            control_boost=float(best_control_boost),
            arthrit_boost=float(best_arthrit_boost),
        )

        model = MultiTaskSegClsModel(
            checkpoint=checkpoint,
            id2label=id2label,
            label2id=label2id,
            class_weights=seg_class_weights_full,
            focal_gamma=best_focal_gamma,
            cls_loss_weight=float(best_cls_loss_weight),
            cls_class_weights=cls_class_weights_full,
            pool_mode=str(pool_mode),
            pool_alpha=float(pool_alpha),
            freeze_encoder=bool(freeze_encoder),
        )

        train_dataset_final = USGSegmentationClassificationDataset(train_paths_final, image_processor, include_cls_labels=True)
        val_dataset_final = USGSegmentationClassificationDataset(val_paths_final, image_processor, include_cls_labels=True)
        test_dataset = USGSegmentationClassificationDataset(fold_test_paths, image_processor, include_cls_labels=True)

        use_fp16 = bool(fp16_if_available) and bool(torch.cuda.is_available())

        training_args = make_training_args(
            output_dir=final_root,
            learning_rate=float(best_lr),
            num_train_epochs=int(final_epochs),
            per_device_train_batch_size=int(best_train_bs),
            per_device_eval_batch_size=int(best_eval_bs),
            evaluation_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_mDice_fg_patient",
            greater_is_better=True,
            logging_dir=os.path.join(final_root, "logs"),
            logging_strategy="epoch",
            seed=int(run_seed),
            dataloader_num_workers=0,
            weight_decay=float(best_wd),
            remove_unused_columns=False,
            report_to="none",
            disable_tqdm=True,
            warmup_ratio=0.1,
            fp16=bool(use_fp16),
            lr_scheduler_type=str(lr_scheduler_type),
            max_grad_norm=float(max_grad_norm),
        )

        epoch_time_cb_final = EpochTimeCallback(tag=f"{fold_name}_{run_tag}")
        es_cb_final = RobustEarlyStoppingCallback(
            metric_key="eval_mDice_fg_patient",
            patience=int(early_patience),
            min_delta=float(early_min_delta),
            smooth_alpha=float(early_smooth_alpha),
            overfit_patience=int(overfit_patience),
            train_delta=float(train_delta),
            val_delta=float(val_delta),
            train_loss_floor=None,
            train_loss_floor_min_epoch=3,
        )

        callbacks = [
            EpochMetricsPrinter(),
            es_cb_final,
            epoch_time_cb_final,
            EncoderUnfreezeCallback(unfreeze_epoch=int(unfreeze_epoch)),
        ]

        trainer = MultiTaskTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset_final,
            eval_dataset=val_dataset_final,
            compute_metrics=compute_metrics,
            callbacks=callbacks,
            use_patient_balanced_sampling=bool(use_patient_balanced_sampling),
            patient_balanced_seed=int(run_seed),
            freeze_encoder=bool(freeze_encoder),
            unfreeze_epoch=int(unfreeze_epoch),
        )

        train_result = trainer.train()

        history_path_final = os.path.join(final_root, "training_history_final.csv")
        save_log_history(trainer.state.log_history, history_path_final)

        extra_info_final = {
            "fold": str(fold_name),
            "run_idx": int(run_idx + 1),
            "seed": int(run_seed),
            "best_lr": float(best_lr),
            "best_weight_decay": float(best_wd),
            "best_train_batch_size": int(best_train_bs),
            "best_eval_batch_size": int(best_eval_bs),
            "best_cls_loss_weight": float(best_cls_loss_weight),
            "best_focal_gamma": float(best_focal_gamma) if best_focal_gamma is not None else None,
            "best_control_boost": float(best_control_boost),
            "best_arthrit_boost": float(best_arthrit_boost),
            "pool_mode": str(pool_mode),
            "pool_alpha": float(pool_alpha),
            "search_mean_val_mDice_fg_patient": float(best_cfg_mdice_patient),
            "patient_balanced_sampling": bool(use_patient_balanced_sampling),
            "freeze_encoder": bool(freeze_encoder),
            "unfreeze_epoch": int(unfreeze_epoch),
            "lr_scheduler_type": str(lr_scheduler_type),
            "max_grad_norm": float(max_grad_norm),
            "fp16": bool(use_fp16),
        }
        append_log_history_to_global_csv(trainer.state.log_history, fold_global_final_epoch_log, extra_info_final)

        epoch_times_final_path = os.path.join(final_root, "epoch_times_final.csv")
        save_epoch_times(epoch_time_cb_final.epoch_times, epoch_times_final_path)

        final_val_metrics = trainer.evaluate(val_dataset_final)
        with open(os.path.join(final_root, "final_val_metrics.json"), "w") as f:
            json.dump({k: float(v) for k, v in final_val_metrics.items()}, f, indent=2)

        if len(val_paths_final) > 0:
            run_inference_on_test(trainer.model, image_processor, val_paths_final, final_root, subdir_name="val_predictions")

        print(f"Fold {fold_name} {run_tag} evaluating on fold test set (test/darkened)")
        test_metrics = trainer.evaluate(test_dataset)
        with open(os.path.join(final_root, "final_fold_test_metrics.json"), "w") as f:
            json.dump({k: float(v) for k, v in test_metrics.items()}, f, indent=2)

        visualize_examples(trainer.model, image_processor, fold_test_paths, final_root, max_examples=None)
        run_inference_on_test(trainer.model, image_processor, fold_test_paths, final_root, subdir_name="fold_test_predictions")

        if len(external_test_paths) > 0:
            run_inference_on_test(trainer.model, image_processor, external_test_paths, final_root, subdir_name="external_test_predictions")

        train_runtime = train_result.metrics.get("train_runtime")

        runs_summary.append(
            {
                "fold": str(fold_name),
                "run_idx": int(run_idx + 1),
                "seed": int(run_seed),
                "best_lr": float(best_lr),
                "best_weight_decay": float(best_wd),
                "best_train_batch_size": int(best_train_bs),
                "best_eval_batch_size": int(best_eval_bs),
                "best_cls_loss_weight": float(best_cls_loss_weight),
                "best_focal_gamma": float(best_focal_gamma) if best_focal_gamma is not None else None,
                "best_control_boost": float(best_control_boost),
                "best_arthrit_boost": float(best_arthrit_boost),
                "pool_mode": str(pool_mode),
                "pool_alpha": float(pool_alpha),
                "search_mean_val_mDice_fg_patient": float(best_cfg_mdice_patient),
                "final_val_mDice_fg_patient": float(final_val_metrics.get("eval_mDice_fg_patient", 0.0)),
                "final_val_loss": float(final_val_metrics.get("eval_loss", 0.0)),
                "final_val_cls_acc_patient": float(final_val_metrics.get("eval_cls_acc_patient", 0.0)),
                "final_val_cls_f1_patient": float(final_val_metrics.get("eval_cls_f1_patient", 0.0)),
                "final_val_cls_roc_auc_patient": float(final_val_metrics.get("eval_cls_roc_auc_patient", 0.0)),
                "final_val_cls_pr_auc_patient": float(final_val_metrics.get("eval_cls_pr_auc_patient", 0.0)),
                "final_test_mDice_fg_patient": float(test_metrics.get("eval_mDice_fg_patient", 0.0)),
                "final_test_loss": float(test_metrics.get("eval_loss", 0.0)),
                "train_runtime": float(train_runtime) if train_runtime is not None else None,
                "patient_balanced_sampling": bool(use_patient_balanced_sampling),
                "freeze_encoder": bool(freeze_encoder),
                "unfreeze_epoch": int(unfreeze_epoch),
                "lr_scheduler_type": str(lr_scheduler_type),
                "max_grad_norm": float(max_grad_norm),
                "fp16": bool(use_fp16),
            }
        )

        torch.cuda.empty_cache()

    if runs_summary:
        fieldnames = sorted(runs_summary[0].keys())
        summary_path = os.path.join(output_root, "runs_summary.csv")
        with open(summary_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for row in runs_summary:
                writer.writerow(row)

        fold_history = {"fold": str(fold_name), "best_cfg": best_cfg, "runs": runs_summary}
        torch.save(fold_history, os.path.join(output_root, "fold_history.pt"))

    return runs_summary


def discover_folds(folds_root):
    dirs = []
    for d in os.listdir(folds_root):
        p = os.path.join(folds_root, d)
        if os.path.isdir(p) and d.startswith("FOLD_"):
            suffix = d.replace("FOLD_", "")
            try:
                idx = int(suffix)
            except Exception:
                continue
            train_val_usg = os.path.join(p, "train_val", "darkened")
            if os.path.isdir(train_val_usg):
                dirs.append((idx, d, p))
    dirs.sort(key=lambda x: x[0])
    return dirs


def main():
    folds_root = r"folds/folds/"
    output_base_root = r"folds/folds/output"
    external_test_usg_dir = None

    search_epochs = 20
    final_epochs = 200
    val_ratio = 0.15
    seed = 42
    checkpoint = "nvidia/segformer-b0-finetuned-ade-512-512"

    k_inner_splits = 3
    search_n_trials = 10

    search_early_patience = 10
    search_early_min_delta = 1e-4
    search_smooth_alpha = 0.6
    search_overfit_patience = 3
    search_train_delta = 0.0
    search_val_delta = 0.0

    lr_min = 3e-5
    lr_max = 1e-4
    wd_min = 0.0
    wd_max = 0.01

    pool_mode = "mask"
    pool_alpha = 1.0

    train_loss_floor = 0.01
    train_loss_floor_min_epoch = 3

    use_patient_balanced_sampling = True
    freeze_encoder_search = True
    unfreeze_epoch_search = 2
    freeze_encoder_final = True
    unfreeze_epoch_final = 5
    lr_scheduler_type = "cosine"
    max_grad_norm = 1.0
    fp16_if_available = True

    num_runs = 10
    final_early_patience = 20
    final_early_min_delta = 1e-4
    final_smooth_alpha = 0.6
    final_overfit_patience = 3
    final_train_delta = 0.0
    final_val_delta = 0.0

    os.makedirs(output_base_root, exist_ok=True)

    folds = discover_folds(folds_root)
    if len(folds) == 0:
        raise RuntimeError(f"No valid folds found in {folds_root}")

    id2label = {0: "background", 1: "fluid"}
    label2id = {"background": 0, "fluid": 1}

    all_runs = []

    for _, fold_name, fold_root in folds:
        output_root = os.path.join(output_base_root, fold_name)
        os.makedirs(output_root, exist_ok=True)

        train_val_usg_dir = os.path.join(fold_root, "train_val", "darkened")
        if not os.path.isdir(train_val_usg_dir):
            raise RuntimeError(f"Train/val dir not found: {train_val_usg_dir}")

        search_output_root = os.path.join(output_root, "SEARCH")
        search_root = os.path.join(search_output_root, "search")
        os.makedirs(search_root, exist_ok=True)

        print(f"Starting nested hyperparameter search for {fold_name}")

        best_cfg = select_best_hyperparams(
            train_val_usg_dirs=[train_val_usg_dir],
            search_root=search_root,
            val_ratio=val_ratio,
            base_seed=seed,
            search_epochs=search_epochs,
            checkpoint=checkpoint,
            id2label=id2label,
            label2id=label2id,
            k_inner_splits=k_inner_splits,
            early_patience=search_early_patience,
            early_min_delta=search_early_min_delta,
            early_smooth_alpha=search_smooth_alpha,
            overfit_patience=search_overfit_patience,
            train_delta=search_train_delta,
            val_delta=search_val_delta,
            lr_min=lr_min,
            lr_max=lr_max,
            wd_min=wd_min,
            wd_max=wd_max,
            search_n_trials=search_n_trials,
            pool_mode=pool_mode,
            pool_alpha=pool_alpha,
            train_loss_floor=train_loss_floor,
            train_loss_floor_min_epoch=train_loss_floor_min_epoch,
            use_patient_balanced_sampling=bool(use_patient_balanced_sampling),
            freeze_encoder=bool(freeze_encoder_search),
            unfreeze_epoch=int(unfreeze_epoch_search),
            lr_scheduler_type=str(lr_scheduler_type),
            max_grad_norm=float(max_grad_norm),
        )

        with open(os.path.join(search_output_root, "best_hyperparams.json"), "w") as f:
            json.dump(best_cfg, f, indent=2)

        best_cfg_mdice_patient = float(best_cfg.get("mean_val_mDice_fg_patient", 0.0))

        print(f"Starting fold training/evaluation for {fold_name} using fold-specific best hyperparameters")

        fold_runs = run_single_fold(
            fold_root=fold_root,
            output_root=output_root,
            best_cfg=best_cfg,
            best_cfg_mdice_patient=best_cfg_mdice_patient,
            final_epochs=final_epochs,
            val_ratio=val_ratio,
            seed=seed,
            checkpoint=checkpoint,
            external_test_usg_dir=external_test_usg_dir,
            num_runs=num_runs,
            early_patience=final_early_patience,
            early_min_delta=final_early_min_delta,
            early_smooth_alpha=final_smooth_alpha,
            overfit_patience=final_overfit_patience,
            train_delta=final_train_delta,
            val_delta=final_val_delta,
            use_patient_balanced_sampling=bool(use_patient_balanced_sampling),
            freeze_encoder=bool(freeze_encoder_final),
            unfreeze_epoch=int(unfreeze_epoch_final),
            lr_scheduler_type=str(lr_scheduler_type),
            max_grad_norm=float(max_grad_norm),
            fp16_if_available=bool(fp16_if_available),
        )

        all_runs.extend(fold_runs)

    if all_runs:
        fieldnames = sorted(all_runs[0].keys())
        global_summary_path = os.path.join(output_base_root, "all_folds_runs_summary.csv")
        with open(global_summary_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for row in all_runs:
                writer.writerow(row)


if __name__ == "__main__":
    main()


[I 2026-02-20 11:53:21,354] A new study created in memory with name: no-name-f9299acd-6f4b-4e02-97a5-a74b6c0ba33b
[W 2026-02-20 11:53:21,377] Trial 0 failed with parameters: {'learning_rate': 6.654411385918238e-05, 'weight_decay': 0.006914950650784979, 'train_batch_size': 2, 'eval_batch_size': 4, 'cls_loss_weight': 0.29000552613081937, 'use_focal_loss': True, 'focal_gamma': 2.8897133189260904, 'control_boost': 2.3257768192966815, 'arthrit_boost': 0.9044582629067428} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/zera/miniconda3/envs/torch_env/lib/python3.11/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_77267/3562248754.py", line 1702, in objective
    mdice_patient, train_runtime, floor_triggered, min_train_loss_seen = train_one_config(
                                                                         ^^^^^^^^^^^^^^

Starting nested hyperparameter search for FOLD_1
Train base patients: 66, images: 666, control: 35, arthrit: 31
Val base patients: 13, images: 129, control: 7, arthrit: 6


In [6]:
import os
import glob
import random
import csv
import json
import time
from collections import Counter
import numpy as np
from PIL import Image
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoImageProcessor, SegformerForSemanticSegmentation, TrainingArguments, Trainer, TrainerCallback
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve, auc, f1_score
import optuna

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

NUM_SEG_LABELS = 2
NUM_CLS_LABELS = 2
CLASSIFICATION_IMAGE_SIZE = 224


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def list_image_files(dir_path):
    exts = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
    out = []
    for p in sorted(glob.glob(os.path.join(dir_path, "*"))):
        if os.path.isfile(p) and p.lower().endswith(exts):
            out.append(p)
    return out


def get_patient_id_from_path(image_path):
    base = os.path.basename(image_path)
    name, _ = os.path.splitext(base)
    if name.startswith("image_"):
        return name[len("image_") :]
    return name


def get_patient_group_from_id(pid):
    if "C" in pid:
        return "control"
    if "P" in pid:
        return "arthrit"
    return "unknown"


def get_patient_base_id(pid):
    if "_" in pid:
        return pid.rsplit("_", 1)[0]
    return pid


def get_cls_label_from_path(image_path):
    pid = get_patient_id_from_path(image_path)
    g = get_patient_group_from_id(pid)
    if g == "control":
        return 0
    if g == "arthrit":
        return 1
    return -100


def resolve_mask_path(image_path):
    img_dir = os.path.dirname(image_path)
    parent = os.path.dirname(img_dir)
    mask_dir = os.path.join(parent, "roi")
    base = os.path.basename(image_path)
    name, ext = os.path.splitext(base)
    if name.startswith("image_"):
        suffix = name[len("image_") :]
        mask_name = "mask_" + suffix + ext
    else:
        if base.startswith("image"):
            mask_name = "mask" + base[5:]
        else:
            mask_name = "mask_" + base
    mask_path = os.path.join(mask_dir, mask_name)
    if not os.path.exists(mask_path):
        raise FileNotFoundError(f"Mask not found for {image_path}: {mask_path}")
    return mask_path


def load_mask_array(image_path):
    mask_path = resolve_mask_path(image_path)
    mask = Image.open(mask_path)
    mask = np.array(mask)
    if mask.ndim == 3:
        mask = mask[:, :, 0]
    if mask.max() > 1:
        mask = (mask > 0).astype("int64")
    else:
        mask = mask.astype("int64")
    return mask


class USGSegmentationClassificationDataset(Dataset):
    def __init__(self, image_paths, processor, include_cls_labels=False):
        self.image_paths = image_paths
        self.processor = processor
        self.include_cls_labels = include_cls_labels

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        mask = load_mask_array(image_path)
        encoded = self.processor(images=image, segmentation_maps=[mask], return_tensors="pt")
        pixel_values = encoded["pixel_values"].squeeze(0)
        labels = encoded["labels"].squeeze(0)
        item = {"pixel_values": pixel_values, "labels": labels}
        if self.include_cls_labels:
            cls_label = get_cls_label_from_path(image_path)
            item["cls_labels"] = torch.tensor(cls_label, dtype=torch.long)
        return item


def stratified_split_base_ids(base_ids, val_ratio, rng):
    control_ids = []
    arthrit_ids = []
    other_ids = []
    for bid in base_ids:
        g = get_patient_group_from_id(bid)
        if g == "control":
            control_ids.append(bid)
        elif g == "arthrit":
            arthrit_ids.append(bid)
        else:
            other_ids.append(bid)
    rng.shuffle(control_ids)
    rng.shuffle(arthrit_ids)
    rng.shuffle(other_ids)

    def split_group(ids):
        if len(ids) <= 1:
            return ids, []
        split_index = int(len(ids) * (1.0 - val_ratio))
        if split_index <= 0:
            split_index = 1
        if split_index >= len(ids):
            split_index = len(ids) - 1
        train_g = ids[:split_index]
        val_g = ids[split_index:]
        return train_g, val_g

    train_c, val_c = split_group(control_ids)
    train_a, val_a = split_group(arthrit_ids)
    train_o, val_o = split_group(other_ids)
    train_ids = train_c + train_a + train_o
    val_ids = val_c + val_a + val_o
    stats = {
        "train_control": len(train_c),
        "train_arthrit": len(train_a),
        "val_control": len(val_c),
        "val_arthrit": len(val_a),
    }
    return train_ids, val_ids, stats


def split_patients(train_val_usg_dir, val_ratio, seed):
    if not os.path.isdir(train_val_usg_dir):
        raise RuntimeError(f"Directory not found: {train_val_usg_dir}")
    all_image_paths = list_image_files(train_val_usg_dir)
    if len(all_image_paths) == 0:
        raise RuntimeError(f"No images found in {train_val_usg_dir}")

    base_to_images = {}
    for p in all_image_paths:
        pid = get_patient_id_from_path(p)
        bid = get_patient_base_id(pid)
        base_to_images.setdefault(bid, []).append(p)

    base_ids = list(base_to_images.keys())
    rng = np.random.RandomState(seed)
    train_bids, val_bids, stats = stratified_split_base_ids(base_ids, val_ratio, rng)

    if not set(train_bids).isdisjoint(set(val_bids)):
        raise RuntimeError("Base patient leakage between train and val sets")

    train_paths = []
    val_paths = []
    for bid in train_bids:
        train_paths.extend(base_to_images[bid])
    for bid in val_bids:
        val_paths.extend(base_to_images[bid])

    print(
        f"Train base patients: {len(train_bids)}, images: {len(train_paths)}, "
        f"control: {stats['train_control']}, arthrit: {stats['train_arthrit']}"
    )
    print(
        f"Val base patients: {len(val_bids)}, images: {len(val_paths)}, "
        f"control: {stats['val_control']}, arthrit: {stats['val_arthrit']}"
    )

    return train_paths, val_paths


def get_fold_test_image_paths(fold_root):
    test_usg_dir = os.path.join(fold_root, "test", "darkened")
    test_roi_dir = os.path.join(fold_root, "test", "roi")
    if not os.path.isdir(test_usg_dir):
        raise RuntimeError(f"Test images dir not found: {test_usg_dir}")
    if not os.path.isdir(test_roi_dir):
        raise RuntimeError(f"Test ROI dir not found: {test_roi_dir}")
    all_image_paths = list_image_files(test_usg_dir)
    if len(all_image_paths) == 0:
        raise RuntimeError(f"No test images found in {test_usg_dir}")
    return all_image_paths


def get_external_test_image_paths(external_test_usg_dir):
    if external_test_usg_dir is None:
        return []
    if not os.path.isdir(external_test_usg_dir):
        return []
    all_image_paths = list_image_files(external_test_usg_dir)
    return all_image_paths


def make_patient_balanced_sampler(image_paths, seed):
    pids = []
    for p in image_paths:
        pid = get_patient_id_from_path(p)
        pids.append(get_patient_base_id(pid))
    cnt = Counter(pids)
    weights = torch.tensor([1.0 / float(cnt[pid]) for pid in pids], dtype=torch.double)
    g = torch.Generator()
    g.manual_seed(int(seed))
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True, generator=g)


def compute_metrics(eval_pred):
    if hasattr(eval_pred, "predictions"):
        preds = eval_pred.predictions
        labels = eval_pred.label_ids
    else:
        preds, labels = eval_pred

    seg_logits_np = None
    cls_logits_np = None
    if isinstance(preds, (tuple, list)):
        for p in preds:
            if isinstance(p, np.ndarray) and getattr(p, "ndim", 0) == 4:
                seg_logits_np = p
            if isinstance(p, np.ndarray) and getattr(p, "ndim", 0) == 2:
                cls_logits_np = p
    else:
        seg_logits_np = preds

    seg_labels_np = None
    cls_labels_np = None
    if isinstance(labels, (tuple, list)):
        for l in labels:
            if isinstance(l, np.ndarray) and getattr(l, "ndim", 0) >= 3:
                seg_labels_np = l
            if isinstance(l, np.ndarray) and getattr(l, "ndim", 0) == 1:
                cls_labels_np = l
    else:
        seg_labels_np = labels

    out = {}

    if seg_logits_np is not None and seg_labels_np is not None:
        logits = torch.as_tensor(seg_logits_np)
        logits = torch.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0).clamp(-50.0, 50.0)
        labels_t = torch.as_tensor(seg_labels_np).long()

        upsampled_logits = F.interpolate(
            logits,
            size=labels_t.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        pred_mask = upsampled_logits.argmax(dim=1)

        valid = labels_t != 255
        pred_v = pred_mask[valid]
        lab_v = labels_t[valid]

        ious = []
        dices = []
        for label_id in range(NUM_SEG_LABELS):
            p = pred_v == label_id
            t = lab_v == label_id
            inter = (p & t).sum().item()
            union = (p | t).sum().item()
            denom = p.sum().item() + t.sum().item()
            iou = float(inter) / float(union) if union != 0 else 0.0
            dice = (2.0 * float(inter) / float(denom)) if denom != 0 else 0.0
            ious.append(iou)
            dices.append(dice)

        out["mIoU"] = float(np.mean(ious))
        out["mDice"] = float(np.mean(dices))
        out["pixel_accuracy"] = float((pred_v == lab_v).float().mean().item())
        out["iou_class_0"] = float(ious[0])
        out["iou_class_1"] = float(ious[1])
        out["dice_class_0"] = float(dices[0])
        out["dice_class_1"] = float(dices[1])
        out["mIoU_fg"] = float(np.mean(ious[1:])) if NUM_SEG_LABELS > 1 else float(ious[0])
        out["mDice_fg"] = float(np.mean(dices[1:])) if NUM_SEG_LABELS > 1 else float(dices[0])

        y_true_flat = lab_v.view(-1).cpu().numpy()
        y_pred_flat = pred_v.view(-1).cpu().numpy()
        try:
            out["f1_micro"] = float(f1_score(y_true_flat, y_pred_flat, average="micro"))
            out["f1_macro"] = float(f1_score(y_true_flat, y_pred_flat, average="macro"))
            out["f1_fluid"] = float(
                f1_score((y_true_flat == 1).astype(np.int32), (y_pred_flat == 1).astype(np.int32))
            )
        except Exception:
            out["f1_micro"] = 0.0
            out["f1_macro"] = 0.0
            out["f1_fluid"] = 0.0

    if cls_logits_np is not None and cls_labels_np is not None:
        cls_logits_t = torch.as_tensor(cls_logits_np)
        cls_logits_t = torch.nan_to_num(cls_logits_t, nan=0.0, posinf=50.0, neginf=-50.0).clamp(-50.0, 50.0)
        cls_labels_t = torch.as_tensor(cls_labels_np).long()

        valid_c = cls_labels_t != -100
        if valid_c.any().item():
            logits_v = cls_logits_t[valid_c]
            labels_v = cls_labels_t[valid_c]

            probs = torch.softmax(logits_v, dim=1)
            probs = torch.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
            row_sums = probs.sum(dim=1, keepdim=True).clamp_min(1e-12)
            probs = probs / row_sums

            pred = torch.argmax(probs, dim=1)

            y_true = labels_v.cpu().numpy()
            y_pred = pred.cpu().numpy()
            y_score = probs[:, 1].detach().cpu().numpy()

            finite = np.isfinite(y_score)
            y_true_f = y_true[finite]
            y_pred_f = y_pred[finite]
            y_score_f = y_score[finite]

            if len(y_true_f) == 0:
                out["cls_acc"] = 0.0
                out["cls_f1"] = 0.0
                out["cls_roc_auc"] = 0.0
                out["cls_pr_auc"] = 0.0
            else:
                out["cls_acc"] = float((y_pred_f == y_true_f).mean())
                try:
                    out["cls_f1"] = float(f1_score(y_true_f, y_pred_f))
                except Exception:
                    out["cls_f1"] = 0.0

                if len(np.unique(y_true_f)) > 1:
                    try:
                        fpr, tpr, _ = roc_curve(y_true_f, y_score_f)
                        out["cls_roc_auc"] = float(auc(fpr, tpr))
                    except Exception:
                        out["cls_roc_auc"] = 0.0
                    try:
                        prec, rec, _ = precision_recall_curve(y_true_f, y_score_f)
                        out["cls_pr_auc"] = float(auc(rec, prec))
                    except Exception:
                        out["cls_pr_auc"] = 0.0
                else:
                    out["cls_roc_auc"] = 0.0
                    out["cls_pr_auc"] = 0.0
        else:
            out["cls_acc"] = 0.0
            out["cls_f1"] = 0.0
            out["cls_roc_auc"] = 0.0
            out["cls_pr_auc"] = 0.0

    return out



def compute_seg_class_weights(image_paths):
    counts = np.zeros(NUM_SEG_LABELS, dtype=np.float64)
    for p in image_paths:
        m = load_mask_array(p)
        for c in range(NUM_SEG_LABELS):
            counts[c] += (m == c).sum()
    freq = counts / counts.sum()
    weights = 1.0 / (freq + 1e-6)
    weights = weights / weights.sum() * NUM_SEG_LABELS
    return weights.astype(np.float32)


def compute_cls_class_weights(image_paths, control_boost=1.0, arthrit_boost=1.0):
    counts = np.zeros(NUM_CLS_LABELS, dtype=np.float64)
    for p in image_paths:
        lab = get_cls_label_from_path(p)
        if 0 <= lab < NUM_CLS_LABELS:
            counts[lab] += 1.0
    total = counts.sum()
    if total == 0:
        return None
    freq = counts / total
    base_weights = 1.0 / (freq + 1e-6)
    boost = np.array([control_boost, arthrit_boost], dtype=np.float64)
    weights = base_weights * boost
    weights = weights / weights.sum() * NUM_CLS_LABELS
    return weights.astype(np.float32)


def segmentation_loss(logits, labels, class_weights=None, focal_gamma=None):
    logits = F.interpolate(
        logits,
        size=labels.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )
    labels = labels.long()
    valid_mask = labels != 255
    logits = logits.permute(0, 2, 3, 1)[valid_mask]
    labels = labels[valid_mask]
    if class_weights is not None:
        weight = class_weights.to(logits.device)
    else:
        weight = None
    ce_loss = F.cross_entropy(logits, labels, weight=weight, reduction="none")
    ce_loss_mean = ce_loss.mean()

    focal_term = 0.0
    if focal_gamma is not None:
        num_classes = logits.shape[-1]
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)
        targets_one_hot = F.one_hot(labels, num_classes=num_classes).float()
        pt = (probs * targets_one_hot).sum(dim=-1)
        focal_term = ((1.0 - pt) ** focal_gamma * ce_loss).mean()

    num_classes = logits.shape[-1]
    probs = torch.softmax(logits, dim=-1)
    targets_one_hot = F.one_hot(labels, num_classes=num_classes).float()
    probs_flat = probs.view(-1, num_classes)
    targets_flat = targets_one_hot.view(-1, num_classes)
    intersection = (probs_flat * targets_flat).sum(dim=0)
    union = probs_flat.sum(dim=0) + targets_flat.sum(dim=0)
    dice_score = (2.0 * intersection + 1.0) / (union + 1.0)
    dice_loss = 1.0 - dice_score.mean()

    return ce_loss_mean + dice_loss + focal_term


def visualize_examples(model, processor, image_paths, output_dir, max_examples=None):
    if len(image_paths) == 0:
        return
    figures_dir = os.path.join(output_dir, "figures")
    os.makedirs(figures_dir, exist_ok=True)
    device = next(model.parameters()).device
    model.eval()
    subset = image_paths if max_examples is None else image_paths[:max_examples]
    for idx, image_path in enumerate(subset):
        image = Image.open(image_path).convert("RGB")
        mask = load_mask_array(image_path)
        inputs = processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)
        with torch.no_grad():
            outputs = model(pixel_values=pixel_values, labels=None, cls_labels=None)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
        upsampled_logits = F.interpolate(
            logits,
            size=mask.shape,
            mode="bilinear",
            align_corners=False,
        )
        pred = upsampled_logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)
        img_np = np.array(image)
        h, w, _ = img_np.shape
        gt_color = np.zeros_like(img_np)
        gt_color[mask == 1] = np.array([255, 0, 0], dtype=np.uint8)
        pred_color = np.zeros_like(img_np)
        pred_color[pred == 1] = np.array([0, 255, 0], dtype=np.uint8)
        overlay_gt = (0.6 * img_np + 0.4 * gt_color).astype(np.uint8)
        overlay_pred = (0.6 * img_np + 0.4 * pred_color).astype(np.uint8)
        canvas = np.zeros((h, w * 3, 3), dtype=np.uint8)
        canvas[:, :w] = img_np
        canvas[:, w : 2 * w] = overlay_gt
        canvas[:, 2 * w : 3 * w] = overlay_pred
        out_img = Image.fromarray(canvas)
        out_path = os.path.join(figures_dir, f"example_{idx + 1}.png")
        out_img.save(out_path)


class EpochMetricsPrinter(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return control
        if "epoch" not in logs:
            return control
        epoch = logs["epoch"]
        keys = [
            "loss",
            "eval_loss",
            "eval_mIoU",
            "eval_mDice",
            "eval_mIoU_fg",
            "eval_mDice_fg",
            "eval_pixel_accuracy",
            "eval_f1_micro",
            "eval_f1_macro",
            "eval_f1_fluid",
            "eval_cls_acc",
            "eval_cls_f1",
            "eval_cls_roc_auc",
            "eval_cls_pr_auc",
        ]
        parts = []
        for k in keys:
            if k in logs:
                v = logs[k]
                if isinstance(v, (float, int)):
                    parts.append(f"{k}: {v:.4f}")
                else:
                    parts.append(f"{k}: {v}")
        if parts:
            text = " | ".join(parts)
            print(f"Epoch {epoch:.1f} -> {text}")
        return control


class EpochTimeCallback(TrainerCallback):
    def __init__(self, tag):
        self.tag = tag
        self.epoch_start_time = None
        self.epoch_times = []

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start_time = time.time()
        return control

    def on_epoch_end(self, args, state, control, **kwargs):
        if self.epoch_start_time is not None:
            duration = time.time() - self.epoch_start_time
            self.epoch_times.append(
                {
                    "tag": self.tag,
                    "epoch": float(state.epoch) if state.epoch is not None else None,
                    "epoch_time": float(duration),
                }
            )
        return control


class RobustEarlyStoppingCallback(TrainerCallback):
    def __init__(
        self,
        metric_key="eval_mDice_fg",
        patience=10,
        min_delta=0.0,
        smooth_alpha=0.6,
        overfit_patience=3,
        train_delta=0.0,
        val_delta=0.0,
        train_loss_floor=None,
        train_loss_floor_min_epoch=3,
    ):
        self.metric_key = metric_key
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.smooth_alpha = float(smooth_alpha)
        self.overfit_patience = int(overfit_patience)
        self.train_delta = float(train_delta)
        self.val_delta = float(val_delta)
        self.train_loss_floor = None if train_loss_floor is None else float(train_loss_floor)
        self.train_loss_floor_min_epoch = int(train_loss_floor_min_epoch)

        self.best_ema = None
        self.bad_epochs = 0
        self.ema_metric = None
        self.ema_val_loss = None

        self.overfit_epochs = 0
        self.epoch_train_losses = {}
        self.history = []
        self.floor_triggered = False
        self.floor_triggered_epoch = None
        self.min_train_loss_seen = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.best_ema = None
        self.bad_epochs = 0
        self.ema_metric = None
        self.ema_val_loss = None
        self.overfit_epochs = 0
        self.epoch_train_losses = {}
        self.history = []
        self.floor_triggered = False
        self.floor_triggered_epoch = None
        self.min_train_loss_seen = None
        return control

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return control
        if "loss" in logs and "epoch" in logs:
            epoch = logs["epoch"]
            try:
                epoch_int = int(round(epoch))
            except Exception:
                epoch_int = int(epoch)
            loss_v = float(logs["loss"])
            self.epoch_train_losses[epoch_int] = loss_v
            if self.min_train_loss_seen is None or loss_v < self.min_train_loss_seen:
                self.min_train_loss_seen = loss_v
            if self.train_loss_floor is not None and epoch_int >= self.train_loss_floor_min_epoch:
                if loss_v < self.train_loss_floor:
                    self.floor_triggered = True
                    self.floor_triggered_epoch = epoch_int
                    control.should_training_stop = True
        return control

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return control
        if "eval_loss" not in metrics or "epoch" not in metrics:
            return control

        epoch = metrics["epoch"]
        try:
            epoch_int = int(round(epoch))
        except Exception:
            epoch_int = int(epoch)

        train_loss = self.epoch_train_losses.get(epoch_int)
        if train_loss is None:
            return control

        val_loss = float(metrics["eval_loss"])
        cur_metric = float(metrics.get(self.metric_key, metrics.get("eval_mDice", 0.0)))

        if self.ema_metric is None:
            self.ema_metric = cur_metric
        else:
            self.ema_metric = self.smooth_alpha * cur_metric + (1.0 - self.smooth_alpha) * self.ema_metric

        if self.ema_val_loss is None:
            self.ema_val_loss = val_loss
        else:
            self.ema_val_loss = self.smooth_alpha * val_loss + (1.0 - self.smooth_alpha) * self.ema_val_loss

        self.history.append(
            {
                "epoch": epoch_int,
                "train_loss": float(train_loss),
                "val_loss": float(val_loss),
                "ema_metric": float(self.ema_metric),
                "ema_val_loss": float(self.ema_val_loss),
            }
        )

        if self.best_ema is None or self.ema_metric > self.best_ema + self.min_delta:
            self.best_ema = float(self.ema_metric)
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1

        if len(self.history) >= 2:
            prev = self.history[-2]
            prev_train = float(prev["train_loss"])
            prev_val = float(prev["val_loss"])
            if float(train_loss) < prev_train - self.train_delta and float(val_loss) > prev_val + self.val_delta:
                self.overfit_epochs += 1
            else:
                self.overfit_epochs = 0

        stop_for_patience = self.bad_epochs >= self.patience
        stop_for_overfit = self.overfit_epochs >= self.overfit_patience

        if stop_for_patience or stop_for_overfit:
            control.should_training_stop = True

        return control


class EncoderUnfreezeCallback(TrainerCallback):
    def __init__(self, unfreeze_epoch=5):
        self.unfreeze_epoch = int(unfreeze_epoch)
        self.done = False

    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        if self.done:
            return control
        ep = state.epoch
        if ep is None:
            return control
        if float(ep) >= float(self.unfreeze_epoch):
            if model is not None and hasattr(model, "set_encoder_trainable"):
                model.set_encoder_trainable(True)
            self.done = True
        return control


def save_log_history(log_history, out_path):
    if not log_history:
        return
    keys = set()
    for entry in log_history:
        keys.update(entry.keys())
    fieldnames = sorted(keys)
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for entry in log_history:
            row = {}
            for k in fieldnames:
                v = entry.get(k)
                if isinstance(v, (int, float, str)) or v is None:
                    row[k] = v
                else:
                    row[k] = str(v)
            writer.writerow(row)


def save_epoch_times(epoch_times, out_path):
    if not epoch_times:
        return
    fieldnames = sorted(epoch_times[0].keys())
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in epoch_times:
            writer.writerow(row)


def append_log_history_to_global_csv(log_history, out_path, extra_info=None):
    if not log_history:
        return
    keys = set()
    for entry in log_history:
        keys.update(entry.keys())
    if extra_info is not None:
        keys.update(extra_info.keys())
    fieldnames = sorted(keys)
    file_exists = os.path.exists(out_path)
    with open(out_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        for entry in log_history:
            row = {}
            for k in fieldnames:
                if extra_info is not None and k in extra_info:
                    v = extra_info[k]
                else:
                    v = entry.get(k)
                if isinstance(v, (int, float, str)) or v is None:
                    row[k] = v
                else:
                    row[k] = str(v)
            writer.writerow(row)


class MultiTaskSegClsModel(nn.Module):
    def __init__(
        self,
        checkpoint,
        id2label,
        label2id,
        class_weights=None,
        focal_gamma=None,
        cls_loss_weight=1.0,
        cls_class_weights=None,
        pool_mode="mask",
        pool_alpha=1.0,
        freeze_encoder=False,
    ):
        super().__init__()
        self.seg_model = SegformerForSemanticSegmentation.from_pretrained(
            checkpoint,
            num_labels=NUM_SEG_LABELS,
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True,
        )
        hidden_dim = self.seg_model.config.hidden_sizes[-1]
        self.cls_classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, NUM_CLS_LABELS),
        )

        self.pool_mode = pool_mode
        self.pool_alpha = float(pool_alpha)

        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        else:
            self.class_weights = None

        if cls_class_weights is not None:
            self.cls_class_weights = torch.tensor(cls_class_weights, dtype=torch.float32)
        else:
            self.cls_class_weights = None

        self.focal_gamma = focal_gamma
        self.cls_loss_weight = cls_loss_weight
        self._freeze_encoder_initial = bool(freeze_encoder)

    def set_encoder_trainable(self, trainable):
        if hasattr(self.seg_model, "segformer"):
            for p in self.seg_model.segformer.parameters():
                p.requires_grad = bool(trainable)

    def freeze_encoder_if_needed(self):
        if self._freeze_encoder_initial:
            self.set_encoder_trainable(False)

    def forward(self, pixel_values=None, labels=None, cls_labels=None):
        outputs_seg = self.seg_model(pixel_values=pixel_values, output_hidden_states=True, return_dict=True)
        seg_logits = outputs_seg.logits
        last_feat = outputs_seg.hidden_states[-1]

        pooled_global = F.adaptive_avg_pool2d(last_feat, output_size=1).view(last_feat.size(0), -1)

        if self.pool_mode == "mask":
            probs = torch.softmax(seg_logits, dim=1)[:, 1:2]
            mask_ds = F.interpolate(probs, size=last_feat.shape[-2:], mode="bilinear", align_corners=False)
            denom = mask_ds.sum(dim=(2, 3)).clamp_min(1e-6)
            pooled_mask = (last_feat * mask_ds).sum(dim=(2, 3)) / denom
            pooled = self.pool_alpha * pooled_mask + (1.0 - self.pool_alpha) * pooled_global
        else:
            pooled = pooled_global

        cls_logits = self.cls_classifier(pooled)

        loss = None
        if labels is not None:
            loss_seg = segmentation_loss(seg_logits, labels, self.class_weights, self.focal_gamma)
            loss = loss_seg
        if cls_labels is not None:
            cls_labels = cls_labels.to(cls_logits.device)
            w = self.cls_class_weights.to(cls_logits.device) if self.cls_class_weights is not None else None
            loss_cls = F.cross_entropy(cls_logits, cls_labels, weight=w, ignore_index=-100)
            loss = (self.cls_loss_weight * loss_cls) if loss is None else (loss + self.cls_loss_weight * loss_cls)

        if loss is None:
            return {"loss": None, "logits": seg_logits, "cls_logits": cls_logits}
        return {"loss": loss, "logits": seg_logits, "cls_logits": cls_logits}


class MultiTaskTrainer(Trainer):
    def __init__(
        self,
        *args,
        use_patient_balanced_sampling=False,
        patient_balanced_seed=0,
        freeze_encoder=False,
        unfreeze_epoch=5,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.use_patient_balanced_sampling = bool(use_patient_balanced_sampling)
        self.patient_balanced_seed = int(patient_balanced_seed)
        self.freeze_encoder = bool(freeze_encoder)
        self.unfreeze_epoch = int(unfreeze_epoch)
        self._encoder_frozen_applied = False

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss
        if return_outputs:
            return loss, outputs
        return loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        has_seg_labels = "labels" in inputs and inputs["labels"] is not None
        has_cls_labels = "cls_labels" in inputs and inputs["cls_labels"] is not None

        inputs = self._prepare_inputs(inputs)
        seg_labels = inputs["labels"] if has_seg_labels else None
        cls_labels = inputs["cls_labels"] if has_cls_labels else None

        with torch.no_grad():
            outputs = model(**inputs)

        if isinstance(outputs, dict):
            loss = outputs.get("loss", None)
            seg_logits = outputs.get("logits", None)
            cls_logits = outputs.get("cls_logits", None)
        else:
            loss = getattr(outputs, "loss", None)
            seg_logits = getattr(outputs, "logits", None)
            cls_logits = None

        if prediction_loss_only:
            return (loss, None, None)

        preds = (seg_logits, cls_logits)
        labs = (seg_labels, cls_labels)
        return (loss, preds, labs)

    def create_optimizer_and_scheduler(self, num_training_steps):
        super().create_optimizer_and_scheduler(num_training_steps)
        if self.freeze_encoder and (not self._encoder_frozen_applied):
            if hasattr(self.model, "freeze_encoder_if_needed"):
                self.model.freeze_encoder_if_needed()
            self._encoder_frozen_applied = True

    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")
        if self.use_patient_balanced_sampling:
            if hasattr(self.train_dataset, "image_paths"):
                sampler = make_patient_balanced_sampler(self.train_dataset.image_paths, self.patient_balanced_seed)
                return DataLoader(
                    self.train_dataset,
                    batch_size=self.args.train_batch_size,
                    sampler=sampler,
                    collate_fn=self.data_collator,
                    num_workers=self.args.dataloader_num_workers,
                    pin_memory=self.args.dataloader_pin_memory,
                    drop_last=self.args.dataloader_drop_last,
                )
        return super().get_train_dataloader()


def plot_confusion(cm, classes, title, save_path):
    fig, ax = plt.subplots()
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    fig.colorbar(im)
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(int(cm[i, j]), "d"), ha="center", va="center")
    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")
    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)


def plot_roc_curve(fpr, tpr, roc_auc, title, save_path):
    fig, ax = plt.subplots()
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)


def plot_pr_curve(precision, recall, pr_auc, title, save_path):
    fig, ax = plt.subplots()
    ax.plot(recall, precision, label=f"AUC = {pr_auc:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(title)
    ax.legend(loc="lower left")
    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)


def run_inference_on_test(model, processor, image_paths, output_root, subdir_name="test_predictions"):
    if len(image_paths) == 0:
        return
    device = next(model.parameters()).device
    model.eval()
    pred_dir = os.path.join(output_root, subdir_name)
    os.makedirs(pred_dir, exist_ok=True)
    csv_path = os.path.join(pred_dir, "predictions.csv")

    rows = []
    total_cls = 0
    correct_cls = 0
    tp_cls = 0
    tn_cls = 0
    fp_cls = 0
    fn_cls = 0
    y_true = []
    y_pred_cls = []
    y_score_cls = []
    patient_cls_votes = {}

    seg_cm = np.zeros((NUM_SEG_LABELS, NUM_SEG_LABELS), dtype=np.int64)
    seg_intersections = np.zeros(NUM_SEG_LABELS, dtype=np.float64)
    seg_unions = np.zeros(NUM_SEG_LABELS, dtype=np.float64)
    seg_denoms = np.zeros(NUM_SEG_LABELS, dtype=np.float64)

    t_start = time.time()

    for image_path in image_paths:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)

        with torch.no_grad():
            outputs = model(pixel_values=pixel_values, labels=None, cls_labels=None)

        if isinstance(outputs, dict):
            seg_logits = outputs["logits"]
            cls_logits = outputs["cls_logits"]
        else:
            seg_logits = outputs.logits
            cls_logits = None

        h, w = image.size[1], image.size[0]
        upsampled_logits = F.interpolate(
            seg_logits,
            size=(h, w),
            mode="bilinear",
            align_corners=False,
        )
        pred_mask = upsampled_logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)

        gt_mask = load_mask_array(image_path)
        for cid in range(NUM_SEG_LABELS):
            pred_c = pred_mask == cid
            gt_c = gt_mask == cid
            inter = np.logical_and(pred_c, gt_c).sum()
            uni = np.logical_or(pred_c, gt_c).sum()
            denom = pred_c.sum() + gt_c.sum()
            seg_intersections[cid] += inter
            seg_unions[cid] += uni
            seg_denoms[cid] += denom
        gt_flat = gt_mask.reshape(-1)
        pred_flat = pred_mask.reshape(-1)
        idx = gt_flat * NUM_SEG_LABELS + pred_flat
        cm_img = np.bincount(idx, minlength=NUM_SEG_LABELS * NUM_SEG_LABELS).reshape(NUM_SEG_LABELS, NUM_SEG_LABELS)
        seg_cm += cm_img

        positive_pixels = int((pred_mask == 1).sum())
        total_pixels = int(pred_mask.size)
        positive_ratio = positive_pixels / float(total_pixels) if total_pixels > 0 else 0.0

        cls_head_class = ""
        cls_head_prob = 0.0
        arthrit_prob = None

        if cls_logits is not None:
            probs_cls = torch.softmax(cls_logits, dim=1)
            probs_cls_np = probs_cls[0].detach().cpu().numpy()
            cls_pred_idx = int(np.argmax(probs_cls_np))
            arthrit_prob = float(probs_cls_np[1])
            cls_head_prob = float(probs_cls_np[cls_pred_idx])
            cls_head_class = "arthrit" if cls_pred_idx == 1 else "control"

        img_np = np.array(image)
        mask_uint8 = ((pred_mask == 1).astype(np.uint8)) * 255
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        overlay = img_np.copy()
        if len(contours) > 0:
            cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)
        blended = (0.6 * img_np + 0.4 * overlay).astype(np.uint8)
        out_img = Image.fromarray(blended)

        base = os.path.basename(image_path)
        pred_label = cls_head_class if cls_head_class in ("control", "arthrit") else "unknown"
        out_name = os.path.splitext(base)[0] + f"_pred_{pred_label}.png"
        out_img.save(os.path.join(pred_dir, out_name))

        pid = get_patient_id_from_path(image_path)
        true_group = get_patient_group_from_id(pid)
        patient_base = get_patient_base_id(pid)

        if (
            true_group in ("control", "arthrit")
            and cls_head_class in ("control", "arthrit")
            and arthrit_prob is not None
        ):
            label_int = 1 if true_group == "arthrit" else 0
            y_true.append(label_int)
            y_score_cls.append(arthrit_prob)
            y_pred_cls.append(1 if cls_head_class == "arthrit" else 0)

            total_cls += 1
            if cls_head_class == true_group:
                correct_cls += 1
            if true_group == "arthrit":
                if cls_head_class == "arthrit":
                    tp_cls += 1
                else:
                    fn_cls += 1
            else:
                if cls_head_class == "control":
                    tn_cls += 1
                else:
                    fp_cls += 1

            pc = patient_cls_votes.get(patient_base)
            if pc is None:
                pc = {"true_group": true_group, "arthrit": 0, "control": 0}
            pc[cls_head_class] += 1
            patient_cls_votes[patient_base] = pc

        rows.append(
            [
                image_path,
                pid,
                total_pixels,
                positive_pixels,
                positive_ratio,
                true_group,
                cls_head_class,
                cls_head_prob,
                arthrit_prob if arthrit_prob is not None else 0.0,
            ]
        )

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(
            [
                "image_path",
                "patient_id",
                "total_pixels",
                "positive_pixels",
                "positive_ratio",
                "true_group",
                "cls_head_class",
                "cls_head_prob",
                "arthrit_prob",
            ]
        )
        writer.writerows(rows)

    def safe_div(a, b):
        return a / b if b > 0 else 0.0

    acc_cls = safe_div(correct_cls, total_cls)
    sens_cls = safe_div(tp_cls, tp_cls + fn_cls)
    spec_cls = safe_div(tn_cls, tn_cls + fp_cls)
    prec_cls = safe_div(tp_cls, tp_cls + fp_cls)
    print(
        f"[{subdir_name}] SegFormer head (image): n={total_cls}, acc={acc_cls:.3f}, sens={sens_cls:.3f}, spec={spec_cls:.3f}, prec={prec_cls:.3f}"
    )

    if len(patient_cls_votes) > 0:
        pc_tp = 0
        pc_tn = 0
        pc_fp = 0
        pc_fn = 0
        for pb, info in patient_cls_votes.items():
            true_group = info["true_group"]
            if info["arthrit"] >= info["control"]:
                pred_group_cls = "arthrit"
            else:
                pred_group_cls = "control"
            if true_group == "arthrit":
                if pred_group_cls == "arthrit":
                    pc_tp += 1
                else:
                    pc_fn += 1
            elif true_group == "control":
                if pred_group_cls == "control":
                    pc_tn += 1
                else:
                    pc_fp += 1
        n_patients_cls = pc_tp + pc_tn + pc_fp + pc_fn
        acc_pc = safe_div(pc_tp + pc_tn, n_patients_cls)
        sens_pc = safe_div(pc_tp, pc_tp + pc_fn)
        spec_pc = safe_div(pc_tn, pc_tn + pc_fp)
        prec_pc = safe_div(pc_tp, pc_tp + pc_fp)
        print(
            f"[{subdir_name}] SegFormer head (patient): n={n_patients_cls}, acc={acc_pc:.3f}, sens={sens_pc:.3f}, spec={spec_pc:.3f}, prec={prec_pc:.3f}"
        )
    else:
        n_patients_cls = 0
        acc_pc = 0.0
        sens_pc = 0.0
        spec_pc = 0.0
        prec_pc = 0.0
        pc_tp = 0
        pc_tn = 0
        pc_fp = 0
        pc_fn = 0

    seg_iou = []
    seg_dice = []
    for cid in range(NUM_SEG_LABELS):
        inter = seg_intersections[cid]
        uni = seg_unions[cid]
        denom = seg_denoms[cid]
        iou_c = float(inter / uni) if uni > 0 else 0.0
        dice_c = float(2.0 * inter / denom) if denom > 0 else 0.0
        seg_iou.append(iou_c)
        seg_dice.append(dice_c)
    mean_seg_iou = float(np.mean(seg_iou)) if len(seg_iou) > 0 else 0.0
    mean_seg_dice = float(np.mean(seg_dice)) if len(seg_dice) > 0 else 0.0

    plot_confusion(
        seg_cm,
        ["background", "fluid"],
        f"{subdir_name} segmentation",
        os.path.join(pred_dir, "confusion_seg.png"),
    )

    roc_auc_c = None
    pr_auc_c = None
    f1_cls_val = None
    cls_cm = None

    if (
        len(y_true) > 0
        and len(set(y_true)) > 1
        and len(y_pred_cls) == len(y_true)
        and len(y_score_cls) == len(y_true)
    ):
        y_true_np = np.array(y_true)
        y_pred_cls_np = np.array(y_pred_cls)
        y_score_cls_np = np.array(y_score_cls)

        cls_cm = confusion_matrix(y_true_np, y_pred_cls_np)
        plot_confusion(
            cls_cm,
            ["control", "arthrit"],
            f"{subdir_name} SegFormer head",
            os.path.join(pred_dir, "confusion_cls.png"),
        )

        fpr_c, tpr_c, _ = roc_curve(y_true_np, y_score_cls_np)
        roc_auc_c = auc(fpr_c, tpr_c)
        plot_roc_curve(
            fpr_c,
            tpr_c,
            roc_auc_c,
            f"{subdir_name} SegFormer head ROC",
            os.path.join(pred_dir, "roc_cls.png"),
        )

        prec_c, rec_c, _ = precision_recall_curve(y_true_np, y_score_cls_np)
        pr_auc_c = auc(rec_c, prec_c)
        plot_pr_curve(
            prec_c,
            rec_c,
            pr_auc_c,
            f"{subdir_name} SegFormer head PR",
            os.path.join(pred_dir, "pr_cls.png"),
        )

        f1_cls_val = f1_score(y_true_np, y_pred_cls_np)

    print(
        f"[{subdir_name}] Segmentation: mIoU={mean_seg_iou:.3f}, mDice={mean_seg_dice:.3f}, IoU0={seg_iou[0]:.3f}, IoU1={seg_iou[1]:.3f}, Dice0={seg_dice[0]:.3f}, Dice1={seg_dice[1]:.3f}"
    )
    if f1_cls_val is not None and roc_auc_c is not None and pr_auc_c is not None:
        print(f"[{subdir_name}] Classification: F1={f1_cls_val:.3f}, ROC AUC={roc_auc_c:.3f}, PR AUC={pr_auc_c:.3f}")

    seg_metrics = {
        "mIoU": mean_seg_iou,
        "mDice": mean_seg_dice,
        "iou_class_0": seg_iou[0] if len(seg_iou) > 0 else 0.0,
        "iou_class_1": seg_iou[1] if len(seg_iou) > 1 else 0.0,
        "dice_class_0": seg_dice[0] if len(seg_dice) > 0 else 0.0,
        "dice_class_1": seg_dice[1] if len(seg_dice) > 1 else 0.0,
    }

    cls_metrics = None
    if f1_cls_val is not None and roc_auc_c is not None and pr_auc_c is not None:
        cls_metrics = {
            "n_images": int(total_cls),
            "acc_image": float(acc_cls),
            "sensitivity_image": float(sens_cls),
            "specificity_image": float(spec_cls),
            "precision_image": float(prec_cls),
            "tp_image": int(tp_cls),
            "tn_image": int(tn_cls),
            "fp_image": int(fp_cls),
            "fn_image": int(fn_cls),
            "n_patients": int(n_patients_cls),
            "acc_patient": float(acc_pc),
            "sensitivity_patient": float(sens_pc),
            "specificity_patient": float(spec_pc),
            "precision_patient": float(prec_pc),
            "tp_patient": int(pc_tp),
            "tn_patient": int(pc_tn),
            "fp_patient": int(pc_fp),
            "fn_patient": int(pc_fn),
            "f1": float(f1_cls_val),
            "roc_auc": float(roc_auc_c),
            "pr_auc": float(pr_auc_c),
        }

    total_inference_time = time.time() - t_start
    inference_time_per_image = total_inference_time / float(len(image_paths)) if len(image_paths) > 0 else 0.0

    metrics_all = {
        "segmentation": seg_metrics,
        "classification": cls_metrics,
        "subdir_name": subdir_name,
        "inference_time_seconds": float(total_inference_time),
        "inference_time_per_image_seconds": float(inference_time_per_image),
    }

    metrics_path = os.path.join(pred_dir, "metrics_summary.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics_all, f, indent=2)


def make_training_args(**kwargs):
    try:
        return TrainingArguments(**kwargs)
    except TypeError:
        if "evaluation_strategy" in kwargs and "eval_strategy" not in kwargs:
            kwargs2 = dict(kwargs)
            kwargs2["eval_strategy"] = kwargs2.pop("evaluation_strategy")
            try:
                return TrainingArguments(**kwargs2)
            except TypeError:
                kwargs = kwargs2
        if "lr_scheduler_type" in kwargs:
            kwargs2 = dict(kwargs)
            kwargs2.pop("lr_scheduler_type", None)
            return TrainingArguments(**kwargs2)
        raise


def train_one_config(
    train_paths,
    val_paths,
    run_output_dir,
    num_epochs,
    train_batch_size,
    eval_batch_size,
    learning_rate,
    weight_decay,
    seed,
    checkpoint,
    id2label,
    label2id,
    focal_gamma,
    cls_loss_weight,
    control_boost,
    arthrit_boost,
    early_patience=10,
    early_min_delta=0.0,
    early_smooth_alpha=0.6,
    overfit_patience=3,
    train_delta=0.0,
    val_delta=0.0,
    global_epoch_log_path=None,
    global_extra_info=None,
    pool_mode="mask",
    pool_alpha=1.0,
    train_loss_floor=None,
    train_loss_floor_min_epoch=3,
    fp16_if_available=True,
    use_patient_balanced_sampling=True,
    freeze_encoder=True,
    unfreeze_epoch=2,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
):
    set_seed(seed)
    os.makedirs(run_output_dir, exist_ok=True)

    image_processor = AutoImageProcessor.from_pretrained(checkpoint, do_reduce_labels=False)
    seg_class_weights = compute_seg_class_weights(train_paths)
    cls_class_weights = compute_cls_class_weights(
        train_paths,
        control_boost=control_boost,
        arthrit_boost=arthrit_boost,
    )

    model = MultiTaskSegClsModel(
        checkpoint=checkpoint,
        id2label=id2label,
        label2id=label2id,
        class_weights=seg_class_weights,
        focal_gamma=focal_gamma,
        cls_loss_weight=cls_loss_weight,
        cls_class_weights=cls_class_weights,
        pool_mode=pool_mode,
        pool_alpha=pool_alpha,
        freeze_encoder=bool(freeze_encoder),
    )

    train_dataset = USGSegmentationClassificationDataset(train_paths, image_processor, include_cls_labels=True)
    val_dataset = USGSegmentationClassificationDataset(val_paths, image_processor, include_cls_labels=True)

    use_fp16 = bool(fp16_if_available) and bool(torch.cuda.is_available())

    training_args = make_training_args(
        output_dir=run_output_dir,
        learning_rate=float(learning_rate),
        num_train_epochs=int(num_epochs),
        per_device_train_batch_size=int(train_batch_size),
        per_device_eval_batch_size=int(eval_batch_size),
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_mDice_fg",
        greater_is_better=True,
        logging_dir=os.path.join(run_output_dir, "logs"),
        logging_strategy="epoch",
        seed=int(seed),
        dataloader_num_workers=0,
        fp16=use_fp16,
        weight_decay=float(weight_decay),
        remove_unused_columns=False,
        report_to="none",
        disable_tqdm=True,
        warmup_ratio=0.1,
        lr_scheduler_type=str(lr_scheduler_type),
        max_grad_norm=float(max_grad_norm),
    )

    epoch_time_cb = EpochTimeCallback(tag="run")
    es_cb = RobustEarlyStoppingCallback(
        metric_key="eval_mDice_fg",
        patience=early_patience,
        min_delta=early_min_delta,
        smooth_alpha=early_smooth_alpha,
        overfit_patience=overfit_patience,
        train_delta=train_delta,
        val_delta=val_delta,
        train_loss_floor=train_loss_floor,
        train_loss_floor_min_epoch=train_loss_floor_min_epoch,
    )

    callbacks = [
        EpochMetricsPrinter(),
        es_cb,
        epoch_time_cb,
        EncoderUnfreezeCallback(unfreeze_epoch=int(unfreeze_epoch)),
    ]

    trainer = MultiTaskTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
        use_patient_balanced_sampling=bool(use_patient_balanced_sampling),
        patient_balanced_seed=int(seed),
        freeze_encoder=bool(freeze_encoder),
        unfreeze_epoch=int(unfreeze_epoch),
    )

    train_result = trainer.train()

    history_path = os.path.join(run_output_dir, "training_history.csv")
    save_log_history(trainer.state.log_history, history_path)

    if global_epoch_log_path is not None:
        append_log_history_to_global_csv(trainer.state.log_history, global_epoch_log_path, global_extra_info)

    epoch_times_path = os.path.join(run_output_dir, "epoch_times.csv")
    save_epoch_times(epoch_time_cb.epoch_times, epoch_times_path)

    metrics_val = trainer.evaluate(val_dataset)
    metrics_all = {}
    metrics_all.update(train_result.metrics)
    metrics_all.update(metrics_val)
    with open(os.path.join(run_output_dir, "metrics_summary.json"), "w") as f:
        json.dump(metrics_all, f, indent=2, default=float)

    mdice = float(metrics_val.get("eval_mDice_fg", metrics_val.get("eval_mDice", 0.0)))
    train_runtime = float(train_result.metrics.get("train_runtime", 0.0))
    min_train_loss_seen = es_cb.min_train_loss_seen if es_cb.min_train_loss_seen is not None else None
    floor_triggered = bool(es_cb.floor_triggered)

    return mdice, train_runtime, floor_triggered, min_train_loss_seen


def select_best_hyperparams(
    train_val_usg_dirs,
    search_root,
    val_ratio,
    base_seed,
    search_epochs,
    checkpoint,
    id2label,
    label2id,
    k_inner_splits=3,
    early_patience=10,
    early_min_delta=0.0,
    early_smooth_alpha=0.6,
    overfit_patience=3,
    train_delta=0.0,
    val_delta=0.0,
    lr_min=3e-5,
    lr_max=1e-4,
    wd_min=0.0,
    wd_max=0.01,
    search_n_trials=20,
    pool_mode="mask",
    pool_alpha=1.0,
    train_loss_floor=0.01,
    train_loss_floor_min_epoch=3,
    use_patient_balanced_sampling=True,
    freeze_encoder=True,
    unfreeze_epoch=2,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
):
    os.makedirs(search_root, exist_ok=True)
    global_search_epoch_log = os.path.join(search_root, "all_search_epochs_log.csv")
    cfg_summaries = []

    def objective(trial):
        lr = trial.suggest_float("learning_rate", float(lr_min), float(lr_max), log=True)
        wd = trial.suggest_float("weight_decay", float(wd_min), float(wd_max))
        train_bs_trial = trial.suggest_categorical("train_batch_size", [1, 2, 4])
        eval_bs_trial = trial.suggest_categorical("eval_batch_size", [1, 2, 4])
        cls_loss_weight_trial = trial.suggest_float("cls_loss_weight", 0.25, 4.0, log=True)
        use_focal = trial.suggest_categorical("use_focal_loss", [False, True])
        focal_gamma_trial = None
        if use_focal:
            focal_gamma_trial = trial.suggest_float("focal_gamma", 0.5, 3.0)
        control_boost_trial = trial.suggest_float("control_boost", 1.0, 3.0)
        arthrit_boost_trial = trial.suggest_float("arthrit_boost", 0.5, 2.0)

        fold_scores = []
        fold_runtimes = []
        fold_floor_hits = 0

        for fold_idx, train_val_usg_dir in enumerate(train_val_usg_dirs):
            split_scores = []
            split_runtimes = []

            for split_idx in range(k_inner_splits):
                split_seed = int(base_seed) + int(fold_idx) * 10000 + int(split_idx)
                train_paths, val_paths = split_patients(train_val_usg_dir, val_ratio, split_seed)

                run_output_dir = os.path.join(
                    search_root, f"trial_{trial.number}_fold_{fold_idx + 1}_split_{split_idx + 1}"
                )

                extra_info = {
                    "trial": trial.number,
                    "fold_idx": fold_idx + 1,
                    "split_idx": split_idx + 1,
                    "learning_rate": float(lr),
                    "weight_decay": float(wd),
                    "train_batch_size": int(train_bs_trial),
                    "eval_batch_size": int(eval_bs_trial),
                    "cls_loss_weight": float(cls_loss_weight_trial),
                    "use_focal_loss": bool(use_focal),
                    "focal_gamma": float(focal_gamma_trial) if focal_gamma_trial is not None else None,
                    "control_boost": float(control_boost_trial),
                    "arthrit_boost": float(arthrit_boost_trial),
                    "seed": int(split_seed),
                    "pool_mode": str(pool_mode),
                    "pool_alpha": float(pool_alpha),
                    "patient_balanced_sampling": bool(use_patient_balanced_sampling),
                    "freeze_encoder": bool(freeze_encoder),
                    "unfreeze_epoch": int(unfreeze_epoch),
                    "lr_scheduler_type": str(lr_scheduler_type),
                    "max_grad_norm": float(max_grad_norm),
                }

                mdice, train_runtime, floor_triggered, min_train_loss_seen = train_one_config(
                    train_paths=train_paths,
                    val_paths=val_paths,
                    run_output_dir=run_output_dir,
                    num_epochs=int(search_epochs),
                    train_batch_size=int(train_bs_trial),
                    eval_batch_size=int(eval_bs_trial),
                    learning_rate=float(lr),
                    weight_decay=float(wd),
                    seed=int(split_seed),
                    checkpoint=checkpoint,
                    id2label=id2label,
                    label2id=label2id,
                    focal_gamma=focal_gamma_trial,
                    cls_loss_weight=float(cls_loss_weight_trial),
                    control_boost=float(control_boost_trial),
                    arthrit_boost=float(arthrit_boost_trial),
                    early_patience=int(early_patience),
                    early_min_delta=float(early_min_delta),
                    early_smooth_alpha=float(early_smooth_alpha),
                    overfit_patience=int(overfit_patience),
                    train_delta=float(train_delta),
                    val_delta=float(val_delta),
                    global_epoch_log_path=global_search_epoch_log,
                    global_extra_info=extra_info,
                    pool_mode=pool_mode,
                    pool_alpha=float(pool_alpha),
                    train_loss_floor=float(train_loss_floor) if train_loss_floor is not None else None,
                    train_loss_floor_min_epoch=int(train_loss_floor_min_epoch),
                    fp16_if_available=True,
                    use_patient_balanced_sampling=bool(use_patient_balanced_sampling),
                    freeze_encoder=bool(freeze_encoder),
                    unfreeze_epoch=int(unfreeze_epoch),
                    lr_scheduler_type=str(lr_scheduler_type),
                    max_grad_norm=float(max_grad_norm),
                )

                if floor_triggered:
                    fold_floor_hits += 1

                cfg_summaries.append(
                    {
                        "trial": trial.number,
                        "fold_idx": fold_idx + 1,
                        "split_idx": split_idx + 1,
                        "learning_rate": float(lr),
                        "weight_decay": float(wd),
                        "train_batch_size": int(train_bs_trial),
                        "eval_batch_size": int(eval_bs_trial),
                        "cls_loss_weight": float(cls_loss_weight_trial),
                        "use_focal_loss": bool(use_focal),
                        "focal_gamma": float(focal_gamma_trial) if focal_gamma_trial is not None else None,
                        "control_boost": float(control_boost_trial),
                        "arthrit_boost": float(arthrit_boost_trial),
                        "pool_mode": str(pool_mode),
                        "pool_alpha": float(pool_alpha),
                        "seed": int(split_seed),
                        "val_mDice_fg": float(mdice),
                        "train_runtime": float(train_runtime),
                        "min_train_loss_seen": float(min_train_loss_seen) if min_train_loss_seen is not None else None,
                        "floor_triggered": bool(floor_triggered),
                        "patient_balanced_sampling": bool(use_patient_balanced_sampling),
                        "freeze_encoder": bool(freeze_encoder),
                        "unfreeze_epoch": int(unfreeze_epoch),
                        "lr_scheduler_type": str(lr_scheduler_type),
                        "max_grad_norm": float(max_grad_norm),
                    }
                )

                split_scores.append(float(mdice))
                split_runtimes.append(float(train_runtime))

                trial.report(float(np.median(split_scores)), step=(fold_idx * k_inner_splits + split_idx + 1))
                if trial.should_prune():
                    raise optuna.TrialPruned()

            fold_score = float(np.median(split_scores)) if len(split_scores) > 0 else 0.0
            fold_runtime = float(np.mean(split_runtimes)) if len(split_runtimes) > 0 else 0.0
            fold_scores.append(fold_score)
            fold_runtimes.append(fold_runtime)

        if train_loss_floor is not None and fold_floor_hits > 0:
            raise optuna.TrialPruned()

        score = float(np.median(fold_scores)) if len(fold_scores) > 0 else 0.0
        mean_runtime = float(np.mean(fold_runtimes)) if len(fold_runtimes) > 0 else 0.0
        trial.set_user_attr("mean_train_runtime", mean_runtime)
        return score

    pruner = optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=0, interval_steps=1)
    study = optuna.create_study(direction="maximize", pruner=pruner)
    study.optimize(objective, n_trials=int(search_n_trials))

    best_trial = study.best_trial
    best_params = best_trial.params

    best_cfg = {
        "learning_rate": float(best_params["learning_rate"]),
        "weight_decay": float(best_params["weight_decay"]),
        "train_batch_size": int(best_params["train_batch_size"]),
        "eval_batch_size": int(best_params["eval_batch_size"]),
        "cls_loss_weight": float(best_params["cls_loss_weight"]),
        "use_focal_loss": bool(best_params["use_focal_loss"]),
        "focal_gamma": float(best_params["focal_gamma"]) if "focal_gamma" in best_params else None,
        "control_boost": float(best_params["control_boost"]),
        "arthrit_boost": float(best_params["arthrit_boost"]),
        "pool_mode": str(pool_mode),
        "pool_alpha": float(pool_alpha),
        "mean_val_mDice_fg": float(best_trial.value),
        "mean_train_runtime": float(best_trial.user_attrs.get("mean_train_runtime", 0.0)),
        "patient_balanced_sampling": bool(use_patient_balanced_sampling),
        "freeze_encoder": bool(freeze_encoder),
        "unfreeze_epoch": int(unfreeze_epoch),
        "lr_scheduler_type": str(lr_scheduler_type),
        "max_grad_norm": float(max_grad_norm),
    }

    if cfg_summaries:
        fieldnames = sorted(cfg_summaries[0].keys())
        csv_path = os.path.join(search_root, "hyperparam_search_summary.csv")
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for row in cfg_summaries:
                writer.writerow(row)

        history_obj = {"candidates": cfg_summaries, "best_cfg": best_cfg}
        torch.save(history_obj, os.path.join(search_root, "hyperparam_history.pt"))

    return best_cfg


def run_single_fold(
    fold_root,
    output_root,
    best_cfg,
    best_cfg_mdice,
    final_epochs,
    val_ratio,
    seed,
    checkpoint,
    external_test_usg_dir=None,
    num_runs=5,
    early_patience=20,
    early_min_delta=1e-4,
    early_smooth_alpha=0.6,
    overfit_patience=3,
    train_delta=0.0,
    val_delta=0.0,
    use_patient_balanced_sampling=True,
    freeze_encoder=True,
    unfreeze_epoch=5,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    fp16_if_available=True,
):
    id2label = {0: "background", 1: "fluid"}
    label2id = {"background": 0, "fluid": 1}
    os.makedirs(output_root, exist_ok=True)

    fold_name = os.path.basename(fold_root.rstrip(os.sep))
    runs_summary = []

    train_val_usg_dir = os.path.join(fold_root, "train_val", "darkened")
    fold_test_paths = get_fold_test_image_paths(fold_root)
    external_test_paths = get_external_test_image_paths(external_test_usg_dir)

    best_lr = float(best_cfg["learning_rate"])
    best_wd = float(best_cfg["weight_decay"])
    best_train_bs = int(best_cfg.get("train_batch_size", 2))
    best_eval_bs = int(best_cfg.get("eval_batch_size", 2))
    best_cls_loss_weight = float(best_cfg.get("cls_loss_weight", 1.0))
    best_focal_gamma = best_cfg.get("focal_gamma", None)
    best_control_boost = float(best_cfg.get("control_boost", 1.0))
    best_arthrit_boost = float(best_cfg.get("arthrit_boost", 1.0))
    pool_mode = best_cfg.get("pool_mode", "mask")
    pool_alpha = float(best_cfg.get("pool_alpha", 1.0))

    fold_global_final_epoch_log = os.path.join(output_root, "all_final_epochs_log.csv")

    for run_idx in range(int(num_runs)):
        run_seed = int(seed) + int(run_idx) * 1000
        run_tag = f"run_{run_idx + 1}"
        run_output_root = os.path.join(output_root, run_tag)
        final_root = os.path.join(run_output_root, "final")
        os.makedirs(final_root, exist_ok=True)

        final_split_seed = int(run_seed) + 999
        train_paths_final, val_paths_final = split_patients(train_val_usg_dir, val_ratio, final_split_seed)

        set_seed(run_seed)
        image_processor = AutoImageProcessor.from_pretrained(checkpoint, do_reduce_labels=False)
        seg_class_weights_full = compute_seg_class_weights(train_paths_final)
        cls_class_weights_full = compute_cls_class_weights(
            train_paths_final,
            control_boost=best_control_boost,
            arthrit_boost=best_arthrit_boost,
        )

        model = MultiTaskSegClsModel(
            checkpoint=checkpoint,
            id2label=id2label,
            label2id=label2id,
            class_weights=seg_class_weights_full,
            focal_gamma=best_focal_gamma,
            cls_loss_weight=best_cls_loss_weight,
            cls_class_weights=cls_class_weights_full,
            pool_mode=pool_mode,
            pool_alpha=pool_alpha,
            freeze_encoder=bool(freeze_encoder),
        )

        train_dataset_final = USGSegmentationClassificationDataset(
            train_paths_final, image_processor, include_cls_labels=True
        )
        val_dataset_final = USGSegmentationClassificationDataset(val_paths_final, image_processor, include_cls_labels=True)
        test_dataset = USGSegmentationClassificationDataset(fold_test_paths, image_processor, include_cls_labels=True)

        use_fp16 = bool(fp16_if_available) and bool(torch.cuda.is_available())

        training_args = make_training_args(
            output_dir=final_root,
            learning_rate=best_lr,
            num_train_epochs=int(final_epochs),
            per_device_train_batch_size=int(best_train_bs),
            per_device_eval_batch_size=int(best_eval_bs),
            evaluation_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_mDice_fg",
            greater_is_better=True,
            logging_dir=os.path.join(final_root, "logs"),
            logging_strategy="epoch",
            seed=int(run_seed),
            dataloader_num_workers=0,
            weight_decay=best_wd,
            remove_unused_columns=False,
            report_to="none",
            disable_tqdm=True,
            warmup_ratio=0.1,
            fp16=use_fp16,
            lr_scheduler_type=str(lr_scheduler_type),
            max_grad_norm=float(max_grad_norm),
        )

        epoch_time_cb_final = EpochTimeCallback(tag=f"{fold_name}_{run_tag}")
        es_cb_final = RobustEarlyStoppingCallback(
            metric_key="eval_mDice_fg",
            patience=int(early_patience),
            min_delta=float(early_min_delta),
            smooth_alpha=float(early_smooth_alpha),
            overfit_patience=int(overfit_patience),
            train_delta=float(train_delta),
            val_delta=float(val_delta),
            train_loss_floor=None,
            train_loss_floor_min_epoch=3,
        )

        callbacks = [
            EpochMetricsPrinter(),
            es_cb_final,
            epoch_time_cb_final,
            EncoderUnfreezeCallback(unfreeze_epoch=int(unfreeze_epoch)),
        ]

        trainer = MultiTaskTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset_final,
            eval_dataset=val_dataset_final,
            compute_metrics=compute_metrics,
            callbacks=callbacks,
            use_patient_balanced_sampling=bool(use_patient_balanced_sampling),
            patient_balanced_seed=int(run_seed),
            freeze_encoder=bool(freeze_encoder),
            unfreeze_epoch=int(unfreeze_epoch),
        )

        train_result = trainer.train()

        history_path_final = os.path.join(final_root, "training_history_final.csv")
        save_log_history(trainer.state.log_history, history_path_final)

        extra_info_final = {
            "fold": fold_name,
            "run_idx": run_idx + 1,
            "seed": run_seed,
            "best_lr": best_lr,
            "best_weight_decay": best_wd,
            "best_train_batch_size": best_train_bs,
            "best_eval_batch_size": best_eval_bs,
            "best_cls_loss_weight": best_cls_loss_weight,
            "best_focal_gamma": best_focal_gamma,
            "best_control_boost": best_control_boost,
            "best_arthrit_boost": best_arthrit_boost,
            "pool_mode": str(pool_mode),
            "pool_alpha": float(pool_alpha),
            "search_mean_val_mDice_fg": float(best_cfg_mdice),
            "patient_balanced_sampling": bool(use_patient_balanced_sampling),
            "freeze_encoder": bool(freeze_encoder),
            "unfreeze_epoch": int(unfreeze_epoch),
            "lr_scheduler_type": str(lr_scheduler_type),
            "max_grad_norm": float(max_grad_norm),
            "fp16": bool(use_fp16),
        }
        append_log_history_to_global_csv(trainer.state.log_history, fold_global_final_epoch_log, extra_info_final)

        epoch_times_final_path = os.path.join(final_root, "epoch_times_final.csv")
        save_epoch_times(epoch_time_cb_final.epoch_times, epoch_times_final_path)

        final_val_metrics = trainer.evaluate(val_dataset_final)
        with open(os.path.join(final_root, "final_val_metrics.json"), "w") as f:
            json.dump({k: float(v) for k, v in final_val_metrics.items()}, f, indent=2)

        final_test_mDice_fg = None
        final_test_loss = None

        if len(val_paths_final) > 0:
            run_inference_on_test(trainer.model, image_processor, val_paths_final, final_root, subdir_name="val_predictions")

        print(f"Fold {fold_name} {run_tag} evaluating on fold test set (test/darkened)")
        test_metrics = trainer.evaluate(test_dataset)
        with open(os.path.join(final_root, "final_fold_test_metrics.json"), "w") as f:
            json.dump({k: float(v) for k, v in test_metrics.items()}, f, indent=2)
        final_test_mDice_fg = test_metrics.get("eval_mDice_fg", test_metrics.get("eval_mDice"))
        final_test_loss = test_metrics.get("eval_loss")
        visualize_examples(trainer.model, image_processor, fold_test_paths, final_root, max_examples=None)
        run_inference_on_test(trainer.model, image_processor, fold_test_paths, final_root, subdir_name="fold_test_predictions")

        if len(external_test_paths) > 0:
            run_inference_on_test(
                trainer.model, image_processor, external_test_paths, final_root, subdir_name="external_test_predictions"
            )

        train_runtime = train_result.metrics.get("train_runtime")

        runs_summary.append(
            {
                "fold": fold_name,
                "run_idx": run_idx + 1,
                "seed": run_seed,
                "best_lr": best_lr,
                "best_weight_decay": best_wd,
                "best_train_batch_size": best_train_bs,
                "best_eval_batch_size": best_eval_bs,
                "best_cls_loss_weight": best_cls_loss_weight,
                "best_focal_gamma": best_focal_gamma,
                "best_control_boost": best_control_boost,
                "best_arthrit_boost": best_arthrit_boost,
                "pool_mode": str(pool_mode),
                "pool_alpha": float(pool_alpha),
                "search_mean_val_mDice_fg": float(best_cfg_mdice),
                "final_val_mDice_fg": float(
                    final_val_metrics.get("eval_mDice_fg", final_val_metrics.get("eval_mDice", 0.0))
                ),
                "final_val_loss": float(final_val_metrics.get("eval_loss", 0.0)),
                "final_val_cls_acc": float(final_val_metrics.get("eval_cls_acc", 0.0))
                if final_val_metrics.get("eval_cls_acc") is not None
                else None,
                "final_val_cls_f1": float(final_val_metrics.get("eval_cls_f1", 0.0))
                if final_val_metrics.get("eval_cls_f1") is not None
                else None,
                "final_val_cls_roc_auc": float(final_val_metrics.get("eval_cls_roc_auc", 0.0))
                if final_val_metrics.get("eval_cls_roc_auc") is not None
                else None,
                "final_val_cls_pr_auc": float(final_val_metrics.get("eval_cls_pr_auc", 0.0))
                if final_val_metrics.get("eval_cls_pr_auc") is not None
                else None,
                "final_test_mDice_fg": float(final_test_mDice_fg) if final_test_mDice_fg is not None else None,
                "final_test_loss": float(final_test_loss) if final_test_loss is not None else None,
                "train_runtime": float(train_runtime) if train_runtime is not None else None,
                "patient_balanced_sampling": bool(use_patient_balanced_sampling),
                "freeze_encoder": bool(freeze_encoder),
                "unfreeze_epoch": int(unfreeze_epoch),
                "lr_scheduler_type": str(lr_scheduler_type),
                "max_grad_norm": float(max_grad_norm),
                "fp16": bool(use_fp16),
            }
        )

        torch.cuda.empty_cache()

    if runs_summary:
        fieldnames = sorted(runs_summary[0].keys())
        summary_path = os.path.join(output_root, "runs_summary.csv")
        with open(summary_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for row in runs_summary:
                writer.writerow(row)

        fold_history = {"fold": fold_name, "best_cfg": best_cfg, "runs": runs_summary}
        torch.save(fold_history, os.path.join(output_root, "fold_history.pt"))

    return runs_summary


def discover_folds(folds_root):
    dirs = []
    for d in os.listdir(folds_root):
        p = os.path.join(folds_root, d)
        if os.path.isdir(p) and d.startswith("FOLD_"):
            suffix = d.replace("FOLD_", "")
            try:
                idx = int(suffix)
            except Exception:
                continue
            train_val_usg = os.path.join(p, "train_val", "darkened")
            if os.path.isdir(train_val_usg):
                dirs.append((idx, d, p))
    dirs.sort(key=lambda x: x[0])
    return dirs


def main():
    folds_root = r"C:\Users\ADMİN\Desktop\artritMAKALE\folds"
    output_base_root = r"C:\Users\ADMİN\Desktop\artritMAKALE\segformer_optuna_hardened"
    external_test_usg_dir = None

    search_epochs = 20
    final_epochs = 200
    val_ratio = 0.15
    seed = 42
    checkpoint = "nvidia/segformer-b0-finetuned-ade-512-512"

    search_fold_count = 1
    k_inner_splits = 3
    search_n_trials = 10

    search_early_patience = 10
    search_early_min_delta = 1e-4
    search_smooth_alpha = 0.6
    search_overfit_patience = 3
    search_train_delta = 0.0
    search_val_delta = 0.0

    lr_min = 3e-5
    lr_max = 1e-4
    wd_min = 0.0
    wd_max = 0.01

    pool_mode = "mask"
    pool_alpha = 1.0

    train_loss_floor = 0.01
    train_loss_floor_min_epoch = 3

    use_patient_balanced_sampling = True
    freeze_encoder_search = True
    unfreeze_epoch_search = 2
    freeze_encoder_final = True
    unfreeze_epoch_final = 5
    lr_scheduler_type = "cosine"
    max_grad_norm = 1.0
    fp16_if_available = True

    num_runs = 10
    final_early_patience = 20
    final_early_min_delta = 1e-4
    final_smooth_alpha = 0.6
    final_overfit_patience = 3
    final_train_delta = 0.0
    final_val_delta = 0.0

    os.makedirs(output_base_root, exist_ok=True)

    folds = discover_folds(folds_root)
    if len(folds) == 0:
        raise RuntimeError(f"No valid folds found in {folds_root}")

    id2label = {0: "background", 1: "fluid"}
    label2id = {"background": 0, "fluid": 1}

    fold1 = [t for t in folds if t[1] == "FOLD_1"]
    if len(fold1) > 0:
        search_folds = fold1[:1]
    else:
        search_folds = folds[:1]

    search_dirs = [os.path.join(p, "train_val", "darkened") for _, _, p in search_folds]
    search_fold_names = [name for _, name, _ in search_folds]

    search_tag = "_".join(search_fold_names)
    search_output_root = os.path.join(output_base_root, f"SEARCH_{search_tag}")
    search_root = os.path.join(search_output_root, "search")
    os.makedirs(search_root, exist_ok=True)

    print(f"Starting hyperparameter search on folds: {search_fold_names}")

    best_cfg = select_best_hyperparams(
        train_val_usg_dirs=search_dirs,
        search_root=search_root,
        val_ratio=val_ratio,
        base_seed=seed,
        search_epochs=search_epochs,
        checkpoint=checkpoint,
        id2label=id2label,
        label2id=label2id,
        k_inner_splits=k_inner_splits,
        early_patience=search_early_patience,
        early_min_delta=search_early_min_delta,
        early_smooth_alpha=search_smooth_alpha,
        overfit_patience=search_overfit_patience,
        train_delta=search_train_delta,
        val_delta=search_val_delta,
        lr_min=lr_min,
        lr_max=lr_max,
        wd_min=wd_min,
        wd_max=wd_max,
        search_n_trials=search_n_trials,
        pool_mode=pool_mode,
        pool_alpha=pool_alpha,
        train_loss_floor=train_loss_floor,
        train_loss_floor_min_epoch=train_loss_floor_min_epoch,
        use_patient_balanced_sampling=bool(use_patient_balanced_sampling),
        freeze_encoder=bool(freeze_encoder_search),
        unfreeze_epoch=int(unfreeze_epoch_search),
        lr_scheduler_type=str(lr_scheduler_type),
        max_grad_norm=float(max_grad_norm),
    )

    best_cfg_mdice = best_cfg["mean_val_mDice_fg"]
    with open(os.path.join(search_output_root, "best_hyperparams.json"), "w") as f:
        json.dump(best_cfg, f, indent=2)

    all_runs = []
    for idx, fold_name, fold_root in folds:
        output_root = os.path.join(output_base_root, fold_name)
        print(f"Starting fold {fold_name} with best hyperparameters from {search_fold_names}")
        fold_runs = run_single_fold(
            fold_root=fold_root,
            output_root=output_root,
            best_cfg=best_cfg,
            best_cfg_mdice=best_cfg_mdice,
            final_epochs=final_epochs,
            val_ratio=val_ratio,
            seed=seed,
            checkpoint=checkpoint,
            external_test_usg_dir=external_test_usg_dir,
            num_runs=num_runs,
            early_patience=final_early_patience,
            early_min_delta=final_early_min_delta,
            early_smooth_alpha=final_smooth_alpha,
            overfit_patience=final_overfit_patience,
            train_delta=final_train_delta,
            val_delta=final_val_delta,
            use_patient_balanced_sampling=bool(use_patient_balanced_sampling),
            freeze_encoder=bool(freeze_encoder_final),
            unfreeze_epoch=int(unfreeze_epoch_final),
            lr_scheduler_type=str(lr_scheduler_type),
            max_grad_norm=float(max_grad_norm),
            fp16_if_available=bool(fp16_if_available),
        )
        all_runs.extend(fold_runs)

    if all_runs:
        fieldnames = sorted(all_runs[0].keys())
        global_summary_path = os.path.join(output_base_root, "all_folds_runs_summary.csv")
        with open(global_summary_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for row in all_runs:
                writer.writerow(row)


if __name__ == "__main__":
    main()


[I 2025-12-18 18:28:56,141] A new study created in memory with name: no-name-333f51b4-396f-4a02-9b86-0d0a998d59d8


Starting hyperparameter search on folds: ['FOLD_1']
Train base patients: 66, images: 666, control: 35, arthrit: 31
Val base patients: 13, images: 129, control: 7, arthrit: 6


  image_processor = cls(**image_processor_dict)
Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b0-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([2]) in the model instantiated
- decode_head.classifier.weight: found shape torch.Size([150, 256, 1, 1]) in the checkpoint and torch.Size([2, 256, 1, 1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1.0 -> loss: 2.5931
{'loss': 2.5931, 'grad_norm': 14.348288536071777, 'learning_rate': 2.4610127903461703e-05, 'epoch': 1.0}
Epoch 1.0 -> eval_loss: 2.4287 | eval_mIoU: 0.4875 | eval_mDice: 0.4938 | eval_mIoU_fg: 0.0001 | eval_mDice_fg: 0.0003 | eval_pixel_accuracy: 0.9748 | eval_f1_micro: 0.9748 | eval_f1_macro: 0.4938 | eval_f1_fluid: 0.0003 | eval_cls_acc: 0.5349 | eval_cls_f1: 0.0000 | eval_cls_roc_auc: 0.5975 | eval_cls_pr_auc: 0.4774
{'eval_loss': 2.4286627769470215, 'eval_mIoU': 0.48745861650379685, 'eval_mDice': 0.4937585748471868, 'eval_pixel_accuracy': 0.9747708439826965, 'eval_iou_class_0': 0.9747707410218772, 'eval_iou_class_1': 0.00014649198571644543, 'eval_dice_class_0': 0.987224208636458, 'eval_dice_class_1': 0.00029294105791561893, 'eval_mIoU_fg': 0.00014649198571644543, 'eval_mDice_fg': 0.00029294105791561893, 'eval_f1_micro': 0.9747708342796149, 'eval_f1_macro': 0.4937585748471868, 'eval_f1_fluid': 0.00029294105791561893, 'eval_cls_acc': 0.5348837209302325, 'eva

  image_processor = cls(**image_processor_dict)
Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b0-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([2]) in the model instantiated
- decode_head.classifier.weight: found shape torch.Size([150, 256, 1, 1]) in the checkpoint and torch.Size([2, 256, 1, 1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch 1.0 -> loss: 1.0928
{'loss': 1.0928, 'grad_norm': inf, 'learning_rate': 3.93910359952211e-06, 'epoch': 1.0}
Epoch 1.0 -> eval_loss: 1.0479 | eval_mIoU: 0.2322 | eval_mDice: 0.3349 | eval_mIoU_fg: 0.0367 | eval_mDice_fg: 0.0708 | eval_pixel_accuracy: 0.4399 | eval_f1_micro: 0.4399 | eval_f1_macro: 0.3349 | eval_f1_fluid: 0.0708 | eval_cls_acc: 0.5154 | eval_cls_f1: 0.0000 | eval_cls_roc_auc: 0.7289 | eval_cls_pr_auc: 0.6176
{'eval_loss': 1.0479222536087036, 'eval_mIoU': 0.23217481400329482, 'eval_mDice': 0.33493699941687116, 'eval_pixel_accuracy': 0.4398803412914276, 'eval_iou_class_0': 0.427676495308474, 'eval_iou_class_1': 0.0366731326981156, 'eval_dice_class_0': 0.5991224156366981, 'eval_dice_class_1': 0.07075158319704423, 'eval_mIoU_fg': 0.0366731326981156, 'eval_mDice_fg': 0.07075158319704423, 'eval_f1_micro': 0.43988034174992485, 'eval_f1_macro': 0.33493699941687116, 'eval_f1_fluid': 0.07075158319704423, 'eval_cls_acc': 0.5153846153846153, 'eval_cls_f1': 0.0, 'eval_cls_roc_a

  image_processor = cls(**image_processor_dict)
Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b0-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([2]) in the model instantiated
- decode_head.classifier.weight: found shape torch.Size([150, 256, 1, 1]) in the checkpoint and torch.Size([2, 256, 1, 1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1.0 -> loss: 1.0698
{'loss': 1.0698, 'grad_norm': 9.29842472076416, 'learning_rate': 3.93910359952211e-06, 'epoch': 1.0}
Epoch 1.0 -> eval_loss: 1.0524 | eval_mIoU: 0.4733 | eval_mDice: 0.4930 | eval_mIoU_fg: 0.0092 | eval_mDice_fg: 0.0182 | eval_pixel_accuracy: 0.9375 | eval_f1_micro: 0.9375 | eval_f1_macro: 0.4930 | eval_f1_fluid: 0.0182 | eval_cls_acc: 0.5038 | eval_cls_f1: 0.1096 | eval_cls_roc_auc: 0.3109 | eval_cls_pr_auc: 0.3511
{'eval_loss': 1.0523860454559326, 'eval_mIoU': 0.4733089701276133, 'eval_mDice': 0.4929572625513097, 'eval_pixel_accuracy': 0.9374653697013855, 'eval_iou_class_0': 0.9374290884972059, 'eval_iou_class_1': 0.009188851758020752, 'eval_dice_class_0': 0.967704153987216, 'eval_dice_class_1': 0.01821037111540352, 'eval_mIoU_fg': 0.009188851758020752, 'eval_mDice_fg': 0.01821037111540352, 'eval_f1_micro': 0.9374653765263449, 'eval_f1_macro': 0.4929572625513097, 'eval_f1_fluid': 0.01821037111540352, 'eval_cls_acc': 0.5038167938931297, 'eval_cls_f1': 0.10958

  image_processor = cls(**image_processor_dict)
Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b0-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([2]) in the model instantiated
- decode_head.classifier.weight: found shape torch.Size([150, 256, 1, 1]) in the checkpoint and torch.Size([2, 256, 1, 1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1.0 -> loss: 1.0788
{'loss': 1.0788, 'grad_norm': 3.6027767658233643, 'learning_rate': 3.93910359952211e-06, 'epoch': 1.0}
Epoch 1.0 -> eval_loss: 1.0658 | eval_mIoU: 0.3953 | eval_mDice: 0.4714 | eval_mIoU_fg: 0.0470 | eval_mDice_fg: 0.0899 | eval_pixel_accuracy: 0.7468 | eval_f1_micro: 0.7468 | eval_f1_macro: 0.4714 | eval_f1_fluid: 0.0899 | eval_cls_acc: 0.5420 | eval_cls_f1: 0.0000 | eval_cls_roc_auc: 0.3998 | eval_cls_pr_auc: 0.4224
{'eval_loss': 1.0657793283462524, 'eval_mIoU': 0.39531662493400876, 'eval_mDice': 0.4713999562242482, 'eval_pixel_accuracy': 0.7467955946922302, 'eval_iou_class_0': 0.743590567753127, 'eval_iou_class_1': 0.04704268211489045, 'eval_dice_class_0': 0.8529417186643226, 'eval_dice_class_1': 0.08985819378417378, 'eval_mIoU_fg': 0.04704268211489045, 'eval_mDice_fg': 0.08985819378417378, 'eval_f1_micro': 0.7467955378175691, 'eval_f1_macro': 0.4713999562242482, 'eval_f1_fluid': 0.08985819378417378, 'eval_cls_acc': 0.5419847328244275, 'eval_cls_f1': 0.0, '

In [4]:
import os
import re
import json
import math
import csv
from datetime import datetime

ROOT_DIR = r"C:\Users\ADMİN\Desktop\artritMAKALE\segformer_optuna_hardened"

def safe_float(x):
    try:
        if x is None:
            return None
        v = float(x)
        if math.isfinite(v):
            return v
        return None
    except Exception:
        return None

def mean_std(values):
    vals = [v for v in values if v is not None and math.isfinite(v)]
    if not vals:
        return None, None
    m = sum(vals) / len(vals)
    if len(vals) < 2:
        return m, 0.0
    var = sum((v - m) ** 2 for v in vals) / (len(vals) - 1)
    return m, math.sqrt(var)

def fmt_mean_pm_sd(m, s, nd=3):
    if m is None or s is None:
        return ""
    return f"{m:.{nd}f}±{s:.{nd}f}"

def parse_fold_run(path):
    parts = path.replace("\\", "/").split("/")
    fold = None
    run = None
    for p in parts:
        if re.fullmatch(r"FOLD_\d+", p):
            fold = p
        if re.fullmatch(r"run_\d+", p):
            run = p
    return fold, run

def read_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def find_files(root, filename):
    out = []
    for d, _, files in os.walk(root):
        if filename in files:
            out.append(os.path.join(d, filename))
    return out

def extract_from_metrics_summary(ms_path):
    data = read_json(ms_path)
    if data.get("subdir_name") != "fold_test_predictions":
        return None
    seg = data.get("segmentation") or {}
    cls = data.get("classification") or {}
    rec = {}
    rec["dice"] = safe_float(seg.get("dice_class_1"))
    rec["iou"] = safe_float(seg.get("iou_class_1"))
    rec["precision"] = safe_float(cls.get("precision_patient"))
    rec["sensitivity"] = safe_float(cls.get("sensitivity_patient"))
    rec["specificity"] = safe_float(cls.get("specificity_patient"))
    rec["acc_patient"] = safe_float(cls.get("acc_patient"))
    rec["roc_auc"] = safe_float(cls.get("roc_auc"))
    rec["f1"] = safe_float(cls.get("f1"))
    return rec

def try_pixel_acc_from_final_eval(ms_path):
    fold_test_dir = os.path.dirname(ms_path)
    final_dir = os.path.dirname(fold_test_dir)
    cand = os.path.join(final_dir, "final_fold_test_metrics.json")
    if not os.path.exists(cand):
        return None
    try:
        d = read_json(cand)
        return safe_float(d.get("eval_pixel_accuracy"))
    except Exception:
        return None

def write_csv(path, header, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        for r in rows:
            w.writerow(r)

def main():
    root = os.path.normpath(os.path.abspath(os.path.expanduser(ROOT_DIR)))
    if not os.path.isdir(root):
        raise RuntimeError(f"Klasör bulunamadı: {root}")

    out_root = os.path.normpath(root.rstrip("\\/") + "_hardened")
    os.makedirs(out_root, exist_ok=True)

    ms_files = find_files(root, "metrics_summary.json")
    if not ms_files:
        raise RuntimeError("metrics_summary.json bulunamadı. ROOT_DIR yanlış olabilir.")

    run_rows = []
    for ms in sorted(ms_files):
        fold, run = parse_fold_run(ms)
        if fold is None or run is None:
            continue
        rec = extract_from_metrics_summary(ms)
        if rec is None:
            continue
        pixel_acc = try_pixel_acc_from_final_eval(ms)
        run_rows.append({
            "Fold": fold,
            "Run": run,
            "Dice": rec["dice"],
            "IoU": rec["iou"],
            "Kesinlik": rec["precision"],
            "Duyarlılık": rec["sensitivity"],
            "PikselAcc": pixel_acc,
            "HastaAcc": rec["acc_patient"],
            "ROC_AUC": rec["roc_auc"],
            "Özgüllük": rec["specificity"],
            "F1": rec["f1"],
        })

    if not run_rows:
        raise RuntimeError("Uygun fold_test_predictions/metrics_summary.json bulunamadı.")

    folds = sorted(set(r["Fold"] for r in run_rows), key=lambda x: int(x.split("_")[1]))
    cols = ["Dice","IoU","Kesinlik","Duyarlılık","PikselAcc","HastaAcc","ROC_AUC","Özgüllük","F1"]

    fold_table = []
    fold_means_for_global = {c: [] for c in cols}

    for f in folds:
        rows_f = [r for r in run_rows if r["Fold"] == f]
        out = {"Fold": f}
        for c in cols:
            m, s = mean_std([rr[c] for rr in rows_f])
            out[c] = m
            out[c + "_SS"] = s
            if m is not None:
                fold_means_for_global[c].append(m)
        fold_table.append(out)

    global_pm = {}
    for c in cols:
        gm, gs = mean_std(fold_means_for_global[c])
        global_pm[c] = fmt_mean_pm_sd(gm, gs, nd=3)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_csv_runs = os.path.join(out_root, f"segformer_runs_{ts}.csv")
    out_csv_folds = os.path.join(out_root, f"segformer_folds_{ts}.csv")

    header_runs = ["Fold","Run","Dice","IoU","Kesinlik","Duyarlılık","PikselAcc","HastaAcc","ROC_AUC","Özgüllük","F1"]
    rows_runs = []
    for r in sorted(run_rows, key=lambda x: (int(x["Fold"].split("_")[1]), int(x["Run"].split("_")[1]))):
        rows_runs.append([r[h] for h in header_runs])
    write_csv(out_csv_runs, header_runs, rows_runs)

    header_folds = ["Fold","Dice","IoU","Kesinlik","Duyarlılık","PikselAcc","HastaAcc","ROC_AUC","Özgüllük","F1","Ort±SS"]
    rows_folds = []
    for fr in fold_table:
        rows_folds.append([
            fr["Fold"],
            fr["Dice"],
            fr["IoU"],
            fr["Kesinlik"],
            fr["Duyarlılık"],
            fr["PikselAcc"],
            fr["HastaAcc"],
            fr["ROC_AUC"],
            fr["Özgüllük"],
            fr["F1"],
            ""
        ])
    rows_folds.append([
        "Ort ± SS",
        global_pm["Dice"],
        global_pm["IoU"],
        global_pm["Kesinlik"],
        global_pm["Duyarlılık"],
        global_pm["PikselAcc"],
        global_pm["HastaAcc"],
        global_pm["ROC_AUC"],
        global_pm["Özgüllük"],
        global_pm["F1"],
        ""
    ])
    write_csv(out_csv_folds, header_folds, rows_folds)

    print(out_root)
    print(out_csv_runs)
    print(out_csv_folds)

if __name__ == "__main__":
    main()


C:\Users\ADMİN\Desktop\artritMAKALE\segformer_optuna_hardened_hardened
C:\Users\ADMİN\Desktop\artritMAKALE\segformer_optuna_hardened_hardened\segformer_runs_20251223_174621.csv
C:\Users\ADMİN\Desktop\artritMAKALE\segformer_optuna_hardened_hardened\segformer_folds_20251223_174621.csv
